In [ ]:
bin\Release\llama-server.exe `
  -m "D:\1\lmstudio-community\Qwen2.5-VL-7B-Instruct-GGUF\Qwen2.5-VL-7B-Instruct-Q4_K_M.gguf" `
  -ngl 99 -c 4096 --port 8080

In [ ]:
pip install smolagents litellm

In [ ]:
from smolagents import CodeAgent, LiteLLMModel, DuckDuckGoSearchTool

# الربط مع سيرفر llama.cpp الذي يعمل على جهازك
# لاحظ أننا نستخدم api_base للإشارة للسيرفر المحلي
model = LiteLLMModel(
    model_id="openai/qwen-local",  # اسم تعريفي فقط
    api_base="http://127.0.0.1:8080/v1", # عنوان السيرفر (تأكد من البورت)
    api_key="none" # السيرفر المحلي لا يحتاج مفتاح
)

# إنشاء العميل (Agent)
agent = CodeAgent(
    tools=[DuckDuckGoSearchTool()],
    model=model,
)

# تشغيل المهمة
result = agent.run("What is the current weather in Paris?")
print(result)

In [ ]:
$env:CMAKE_ARGS="-DGGML_CUDA=ON"
pip install llama-cpp-python smolagents

In [2]:
!pip install llama-cpp-python smolagents

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 MB 11.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.7 MB/s eta 0:00:00
  Created wheel for llama-cpp-python: filename=llama_cpp_python-0.3.34-py3-none-linux_x86_64.whl size=20319392 sha256=e629a426bdd883d585fe04c09997161ac6e97048889974d1c40d7af2ec616cf3
  Stored in directory: /root/.cache/pip/wheels/4a/10/e7/0eb9b120f1640844f33562a3964c5b18b67de1d66d3f9530e8
Successfully built llama-cpp-python


Successfully installed diskcache-5.6.3 llama-cpp-python-0.3.34 smolagents-1.26.0

In [3]:
from smolagents import CodeAgent, DuckDuckGoSearchTool, Model
from llama_cpp import Llama

# 1. بناء "جسر" لتشغيل GGUF داخل smolagents
class ColabGGUFModel(Model):
    def __init__(self, model_path):
        super().__init__()
        # تحميل الموديل على المعالج (n_gpu_layers=0 تعني CPU فقط)
        self.llm = Llama(
            model_path=model_path,
            n_ctx=2048,      # تقليل الذاكرة ليناسب كولاب
            n_threads=2,     # كولاب يوفر عادةً نواتين فقط
            n_gpu_layers=0   # إجبار العمل على المعالج
        )

    def forward(self, messages, stop_sequences=None):
        # تحويل صيغة الرسائل ليفهمها الموديل
        prompt = ""
        for msg in messages:
            role = msg['role']
            content = msg['content']
            prompt += f"<|im_start|>{role}\n{content}<|im_end|>\n"
        prompt += "<|im_start|>assistant\n"

        # توليد الإجابة
        output = self.llm(
            prompt,
            max_tokens=500,
            stop=["<|im_end|>", "User:"],
            echo=False
        )
        return output['choices'][0]['text']

# 2. مسار ملف الـ GGUF (تأكد أنك قمت برفعه على كولاب أو وضعت المسار الصحيح)
# إذا كان الملف في درايف مثلاً: /content/drive/MyDrive/model.gguf
model_path = "/content/Qwen2.5-1.5B-Instruct-Q4_K_M.gguf"

# 3. تشغيل النظام
try:
    local_model = ColabGGUFModel(model_path)

    agent = CodeAgent(
        tools=[DuckDuckGoSearchTool()],
        model=local_model,
    )

    # 4. طلب المهمة
    result = agent.run("ما هي عاصمة فرنسا؟")
    print(result)
except Exception as e:
    print(f"حدث خطأ: {e}")
    print("تأكد من أن مسار ملف الـ GGUF صحيح وأن حجمه لا يتعدى 8GB ليعمل في كولاب.")

llama_model_loader: loaded meta data with 38 key-value pairs and 338 tensors from /content/Qwen2.5-1.5B-Instruct-Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen2
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Qwen2.5 1.5B Instruct
llama_model_loader: - kv   3:                           general.finetune str              = Instruct
llama_model_loader: - kv   4:                           general.basename str              = Qwen2.5
llama_model_loader: - kv   5:                         general.size_label str              = 1.5B
llama_model_loader: - kv   6:                            general.license str              = apache-2.0
llama_model_loader: - kv   7:          

حدث خطأ: You must install package `ddgs` to run this tool: for instance run `pip install ddgs`.
تأكد من أن مسار ملف الـ GGUF صحيح وأن حجمه لا يتعدى 8GB ليعمل في كولاب.


CPU : SSE3 = 1 | SSSE3 = 1 | AVX = 1 | AVX2 = 1 | F16C = 1 | FMA = 1 | BMI2 = 1 | LLAMAFILE = 1 | OPENMP = 1 | REPACK = 1 | 
Model metadata: {'quantize.imatrix.chunks_count': '128', 'quantize.imatrix.entries_count': '196', 'quantize.imatrix.file': '/models_out/Qwen2.5-1.5B-Instruct-GGUF/Qwen2.5-1.5B-Instruct.imatrix', 'general.base_model.0.repo_url': 'https://huggingface.co/Qwen/Qwen2.5-1.5B', 'general.license': 'apache-2.0', 'qwen2.attention.head_count_kv': '2', 'tokenizer.chat_template': '{%- if tools %}\n    {{- \'<|im_start|>system\\n\' }}\n    {%- if messages[0][\'role\'] == \'system\' %}\n        {{- messages[0][\'content\'] }}\n    {%- else %}\n        {{- \'You are Qwen, created by Alibaba Cloud. You are a helpful assistant.\' }}\n    {%- endif %}\n    {{- "\\n\\n# Tools\\n\\nYou may call one or more functions to assist with the user query.\\n\\nYou are provided with function signatures within <tools></tools> XML tags:\\n<tools>" }}\n    {%- for tool in tools %}\n        {{- "\

In [1]:
!wget https://huggingface.co/bartowski/Qwen2.5-1.5B-Instruct-GGUF/resolve/main/Qwen2.5-1.5B-Instruct-Q4_K_M.gguf

--2026-08-06 23:18:25--  https://huggingface.co/bartowski/Qwen2.5-1.5B-Instruct-GGUF/resolve/main/Qwen2.5-1.5B-Instruct-Q4_K_M.gguf
Resolving huggingface.co (huggingface.co)... 3.170.185.35, 3.170.185.33, 3.170.185.25, ...
Connecting to huggingface.co (huggingface.co)|3.170.185.35|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/66e9d8d17c1e3024b8017d09/7cdb294099b97d05c78e29de4456efe3f09f3e24a59cc1f13ec7abe28f84c306?user_id=public&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27Qwen2.5-1.5B-Instruct-Q4_K_M.gguf%3B+filename%3D%22Qwen2.5-1.5B-Instruct-Q4_K_M.gguf%22%3B&X-Xet-Cas-Uid=public&Expires=1786061905&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjZlOWQ4ZDE3YzFlMzAyNGI4MDE3ZDA5LzdjZGIyOTQwOTliOTdkMDVjNzhlMjlkZTQ0NTZlZmUzZjA5ZjNlMjRhNTljYzFmMTNlYzdhYmUyOGY4NGMzMDZcXD91c2VyX2lkPXB1YmxpYyZyZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPSomWC1YZXQtQ2FzLVVpZD1wdWJ

In [4]:
!pip install ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 74.0 MB/s eta 0:00:00


In [1]:
from smolagents import CodeAgent, DuckDuckGoSearchTool, Model
from llama_cpp import Llama

# 1. بناء "جسر" لتشغيل GGUF داخل smolagents
class ColabGGUFModel(Model):
    def __init__(self, model_path):
        super().__init__()
        # تحميل الموديل على المعالج (n_gpu_layers=0 تعني CPU فقط)
        self.llm = Llama(
            model_path=model_path,
            n_ctx=2048,      # تقليل الذاكرة ليناسب كولاب
            n_threads=2,     # كولاب يوفر عادةً نواتين فقط
            n_gpu_layers=0   # إجبار العمل على المعالج
        )

    def forward(self, messages, stop_sequences=None):
        # تحويل صيغة الرسائل ليفهمها الموديل
        prompt = ""
        for msg in messages:
            role = msg['role']
            content = msg['content']
            prompt += f"<|im_start|>{role}\n{content}<|im_end|>\n"
        prompt += "<|im_start|>assistant\n"

        # توليد الإجابة
        output = self.llm(
            prompt,
            max_tokens=500,
            stop=["<|im_end|>", "User:"],
            echo=False
        )
        return output['choices'][0]['text']

# 2. مسار ملف الـ GGUF (تأكد أنك قمت برفعه على كولاب أو وضعت المسار الصحيح)
# إذا كان الملف في درايف مثلاً: /content/drive/MyDrive/model.gguf
model_path = "/content/Qwen2.5-1.5B-Instruct-Q4_K_M.gguf"

# 3. تشغيل النظام
try:
    local_model = ColabGGUFModel(model_path)

    agent = CodeAgent(
        tools=[DuckDuckGoSearchTool()],
        model=local_model,
    )

    # 4. طلب المهمة
    result = agent.run("ما هي عاصمة فرنسا؟")
    print(result)
except Exception as e:
    print(f"حدث خطأ: {e}")
    print("تأكد من أن مسار ملف الـ GGUF صحيح وأن حجمه لا يتعدى 8GB ليعمل في كولاب.")

llama_model_loader: loaded meta data with 38 key-value pairs and 338 tensors from /content/Qwen2.5-1.5B-Instruct-Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen2
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Qwen2.5 1.5B Instruct
llama_model_loader: - kv   3:                           general.finetune str              = Instruct
llama_model_loader: - kv   4:                           general.basename str              = Qwen2.5
llama_model_loader: - kv   5:                         general.size_label str              = 1.5B
llama_model_loader: - kv   6:                            general.license str              = apache-2.0
llama_model_loader: - kv   7:          

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ ما هي عاصمة فرنسا؟                                                                                              │
│                                                                                                                 │
╰─ ColabGGUFModel - None ─────────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Error in generating model output:
This method must be implemented in child classes

[Step 1: Duration 0.01 seconds]

حدث خطأ: Error in generating model output:
This method must be implemented in child classes
تأكد من أن مسار ملف الـ GGUF صحيح وأن حجمه لا يتعدى 8GB ليعمل في كولاب.


In [2]:
from smolagents import CodeAgent, DuckDuckGoSearchTool, Model
from llama_cpp import Llama

# 1. بناء الجسر بين GGUF و smolagents
class ColabGGUFModel(Model):
    def __init__(self, model_path):
        super().__init__()
        self.llm = Llama(
            model_path=model_path,
            n_ctx=2048,
            n_threads=2, # كولاب يوفر نواتين
            n_gpu_layers=0 # التأكيد على استخدام المعالج فقط
        )

    def forward(self, messages, stop_sequences=None):
        # تحويل صيغة Qwen Chat
        prompt = ""
        for msg in messages:
            role = msg['role']
            content = msg['content']
            prompt += f"<|im_start|>{role}\n{content}<|im_end|>\n"
        prompt += "<|im_start|>assistant\n"

        output = self.llm(
            prompt,
            max_tokens=500,
            stop=["<|im_end|>", "<|endoftext|>"],
            echo=False
        )
        return output['choices'][0]['text']

# 2. مسار الموديل الذي يظهر في اللوج الخاص بك
model_path = "/content/Qwen2.5-1.5B-Instruct-Q4_K_M.gguf"

# 3. تشغيل النظام
try:
    local_model = ColabGGUFModel(model_path)

    # إضافة أداة البحث
    search_tool = DuckDuckGoSearchTool()

    agent = CodeAgent(
        tools=[search_tool],
        model=local_model,
    )

    # 4. طلب المهمة
    # بما أنك على المعالج، انتظر قليلاً فالإجابة ستستغرق وقتاً
    result = agent.run("What is the current weather in Cairo?")
    print(result)

except Exception as e:
    print(f"حدث خطأ: {e}")

llama_model_loader: loaded meta data with 38 key-value pairs and 338 tensors from /content/Qwen2.5-1.5B-Instruct-Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen2
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Qwen2.5 1.5B Instruct
llama_model_loader: - kv   3:                           general.finetune str              = Instruct
llama_model_loader: - kv   4:                           general.basename str              = Qwen2.5
llama_model_loader: - kv   5:                         general.size_label str              = 1.5B
llama_model_loader: - kv   6:                            general.license str              = apache-2.0
llama_model_loader: - kv   7:          

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ What is the current weather in Cairo?                                                                           │
│                                                                                                                 │
╰─ ColabGGUFModel - None ─────────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Error in generating model output:
This method must be implemented in child classes

[Step 1: Duration 0.00 seconds]

حدث خطأ: Error in generating model output:
This method must be implemented in child classes


In [3]:
# 1. تأكد من تثبيت المكتبات اللازمة أولاً
# !pip install llama-cpp-python smolagents duckduckgo-search

from smolagents import CodeAgent, DuckDuckGoSearchTool, Model
from llama_cpp import Llama

# 2. بناء الكلاس المتوافق مع الإصدار الجديد لـ smolagents
class ColabGGUFModel(Model):
    def __init__(self, model_path):
        super().__init__()
        # تحميل الموديل على المعالج
        self.llm = Llama(
            model_path=model_path,
            n_ctx=2048,
            n_threads=2,
            n_gpu_layers=0
        )

    # الميثود المطلوبة الآن هي __call__ بدلاً من forward
    def __call__(self, messages, stop_sequences=None, **kwargs):
        # تحويل المحادثة إلى صيغة Qwen
        prompt = ""
        for msg in messages:
            role = msg['role']
            content = msg['content']
            prompt += f"<|im_start|>{role}\n{content}<|im_end|>\n"
        prompt += "<|im_start|>assistant\n"

        # توليد النص
        output = self.llm(
            prompt,
            max_tokens=500,
            stop=["<|im_end|>", "<|endoftext|>"],
            echo=False
        )

        # استخراج النص الناتج
        generated_text = output['choices'][0]['text']

        # ملاحظة: smolagents يتوقع كائن يحتوي على خاصية content أو نص مباشر
        return generated_text

# 3. تشغيل النظام
model_path = "/content/Qwen2.5-1.5B-Instruct-Q4_K_M.gguf"

try:
    # تهيئة الموديل
    local_model = ColabGGUFModel(model_path)

    # تهيئة العميل (Agent)
    agent = CodeAgent(
        tools=[DuckDuckGoSearchTool()],
        model=local_model,
    )

    # 4. تنفيذ السؤال
    print("جاري التفكير... (قد يستغرق وقتاً على المعالج)")
    result = agent.run("ما هي عاصمة فرنسا؟")
    print(f"\nالنتيجة النهائية: {result}")

except Exception as e:
    print(f"\nحدث خطأ أثناء التشغيل: {e}")

llama_model_loader: loaded meta data with 38 key-value pairs and 338 tensors from /content/Qwen2.5-1.5B-Instruct-Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen2
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Qwen2.5 1.5B Instruct
llama_model_loader: - kv   3:                           general.finetune str              = Instruct
llama_model_loader: - kv   4:                           general.basename str              = Qwen2.5
llama_model_loader: - kv   5:                         general.size_label str              = 1.5B
llama_model_loader: - kv   6:                            general.license str              = apache-2.0
llama_model_loader: - kv   7:          

جاري التفكير... (قد يستغرق وقتاً على المعالج)


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ ما هي عاصمة فرنسا؟                                                                                              │
│                                                                                                                 │
╰─ ColabGGUFModel - None ─────────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Error in generating model output:
This method must be implemented in child classes

[Step 1: Duration 0.00 seconds]


حدث خطأ أثناء التشغيل: Error in generating model output:
This method must be implemented in child classes


In [ ]:
!pip install llama-cpp-python smolagents duckduckgo-search

In [4]:
!pip install duckduckgo-search

In [5]:
from smolagents import CodeAgent, DuckDuckGoSearchTool, Model
from smolagents.models import ChatMessage # إضافة هذا السطر
from llama_cpp import Llama

# 1. بناء الجسر بشكل يتوافق مع التحديثات الجديدة
class ColabGGUFModel(Model):
    def __init__(self, model_path):
        super().__init__()
        self.llm = Llama(
            model_path=model_path,
            n_ctx=2048,
            n_threads=2,
            n_gpu_layers=0
        )

    # التعديل هنا: إضافة **kwargs وتغيير المخرجات
    def forward(self, messages, stop_sequences=None, **kwargs):
        # تحويل صيغة الشات لنموذج Qwen
        prompt = ""
        for msg in messages:
            role = msg['role']
            content = msg['content']
            prompt += f"<|im_start|>{role}\n{content}<|im_end|>\n"
        prompt += "<|im_start|>assistant\n"

        output = self.llm(
            prompt,
            max_tokens=500,
            stop=["<|im_end|>", "<|endoftext|>"],
            echo=False
        )

        generated_text = output['choices'][0]['text']

        # التعديل الهام: يجب إرجاع كائن ChatMessage وليس نصاً مجرداً
        return ChatMessage(role="assistant", content=generated_text)

# 2. مسار الموديل
model_path = "/content/Qwen2.5-1.5B-Instruct-Q4_K_M.gguf"

# 3. تشغيل النظام
try:
    print("جاري تحميل الموديل...")
    local_model = ColabGGUFModel(model_path)

    agent = CodeAgent(
        tools=[DuckDuckGoSearchTool()],
        model=local_model,
    )

    print("جاري تنفيذ المهمة (قد يستغرق وقتاً على المعالج)...")
    result = agent.run("What is the current weather in Cairo?")
    print("\n--- النتيجة النهائية ---")
    print(result)

except Exception as e:
    print(f"\nحدث خطأ: {e}")

llama_model_loader: loaded meta data with 38 key-value pairs and 338 tensors from /content/Qwen2.5-1.5B-Instruct-Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen2
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Qwen2.5 1.5B Instruct
llama_model_loader: - kv   3:                           general.finetune str              = Instruct
llama_model_loader: - kv   4:                           general.basename str              = Qwen2.5
llama_model_loader: - kv   5:                         general.size_label str              = 1.5B
llama_model_loader: - kv   6:                            general.license str              = apache-2.0
llama_model_loader: - kv   7:          

جاري تحميل الموديل...


init_tokenizer: initializing tokenizer for type 2
load: 0 unused tokens
load: control token: 151660 '<|fim_middle|>' is not marked as EOG
load: control token: 151659 '<|fim_prefix|>' is not marked as EOG
load: control token: 151653 '<|vision_end|>' is not marked as EOG
load: control token: 151648 '<|box_start|>' is not marked as EOG
load: control token: 151646 '<|object_ref_start|>' is not marked as EOG
load: control token: 151649 '<|box_end|>' is not marked as EOG
load: control-looking token: 128247 '</s>' was not control-type; this is probably a bug in the model. its type will be overridden
load: control token: 151655 '<|image_pad|>' is not marked as EOG
load: control token: 151651 '<|quad_end|>' is not marked as EOG
load: control token: 151647 '<|object_ref_end|>' is not marked as EOG
load: control token: 151652 '<|vision_start|>' is not marked as EOG
load: control token: 151654 '<|vision_pad|>' is not marked as EOG
load: control token: 151656 '<|video_pad|>' is not marked as EOG
lo

جاري تنفيذ المهمة (قد يستغرق وقتاً على المعالج)...


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ What is the current weather in Cairo?                                                                           │
│                                                                                                                 │
╰─ ColabGGUFModel - None ─────────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Error in generating model output:
This method must be implemented in child classes

[Step 1: Duration 0.00 seconds]


حدث خطأ: Error in generating model output:
This method must be implemented in child classes


In [6]:
# 1. تثبيت المكتبات (تأكد من تشغيل هذا السطر إذا لم تكن فعلت)
# !pip install llama-cpp-python smolagents duckduckgo-search

from smolagents import CodeAgent, DuckDuckGoSearchTool, Model
from smolagents.models import ChatMessage # ضروري جداً للإصدار الجديد
from llama_cpp import Llama

# 2. بناء الكلاس المتوافق مع متطلبات smolagents الحالية
class ColabGGUFModel(Model):
    def __init__(self, model_path):
        super().__init__()
        # تحميل الموديل على المعالج (CPU)
        self.llm = Llama(
            model_path=model_path,
            n_ctx=2048,
            n_threads=2,
            n_gpu_layers=0
        )

    # المكتبة تبحث عن دالة اسمها forward
    def forward(self, messages, stop_sequences=None, **kwargs) -> ChatMessage:
        # بناء الـ Prompt بصيغة Qwen
        prompt = ""
        for msg in messages:
            role = msg['role']
            content = msg['content']
            prompt += f"<|im_start|>{role}\n{content}<|im_end|>\n"
        prompt += "<|im_start|>assistant\n"

        # توليد الرد من GGUF
        output = self.llm(
            prompt,
            max_tokens=512,
            stop=["<|im_end|>", "<|endoftext|>"],
            echo=False
        )

        generated_text = output['choices'][0]['text']

        # التعديل الجوهري: يجب إرجاع كائن ChatMessage
        return ChatMessage(role="assistant", content=generated_text)

# 3. تشغيل النظام
model_path = "/content/Qwen2.5-1.5B-Instruct-Q4_K_M.gguf"

try:
    print("جاري تهيئة الموديل...")
    local_model = ColabGGUFModel(model_path)

    # تهيئة العميل (Agent)
    agent = CodeAgent(
        tools=[DuckDuckGoSearchTool()],
        model=local_model,
    )

    print("بدأ التفكير... الإجابة قد تستغرق دقيقة على المعالج.")
    # تنفيذ السؤال
    result = agent.run("ما هي عاصمة فرنسا؟")

    print("\n" + "="*30)
    print(f"النتيجة النهائية: {result}")
    print("="*30)

except Exception as e:
    print(f"\nحدث خطأ أثناء التشغيل: {e}")

llama_model_loader: loaded meta data with 38 key-value pairs and 338 tensors from /content/Qwen2.5-1.5B-Instruct-Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen2
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Qwen2.5 1.5B Instruct
llama_model_loader: - kv   3:                           general.finetune str              = Instruct
llama_model_loader: - kv   4:                           general.basename str              = Qwen2.5
llama_model_loader: - kv   5:                         general.size_label str              = 1.5B
llama_model_loader: - kv   6:                            general.license str              = apache-2.0
llama_model_loader: - kv   7:          

جاري تهيئة الموديل...


init_tokenizer: initializing tokenizer for type 2
load: 0 unused tokens
load: control token: 151660 '<|fim_middle|>' is not marked as EOG
load: control token: 151659 '<|fim_prefix|>' is not marked as EOG
load: control token: 151653 '<|vision_end|>' is not marked as EOG
load: control token: 151648 '<|box_start|>' is not marked as EOG
load: control token: 151646 '<|object_ref_start|>' is not marked as EOG
load: control token: 151649 '<|box_end|>' is not marked as EOG
load: control-looking token: 128247 '</s>' was not control-type; this is probably a bug in the model. its type will be overridden
load: control token: 151655 '<|image_pad|>' is not marked as EOG
load: control token: 151651 '<|quad_end|>' is not marked as EOG
load: control token: 151647 '<|object_ref_end|>' is not marked as EOG
load: control token: 151652 '<|vision_start|>' is not marked as EOG
load: control token: 151654 '<|vision_pad|>' is not marked as EOG
load: control token: 151656 '<|video_pad|>' is not marked as EOG
lo

بدأ التفكير... الإجابة قد تستغرق دقيقة على المعالج.


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ ما هي عاصمة فرنسا؟                                                                                              │
│                                                                                                                 │
╰─ ColabGGUFModel - None ─────────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Error in generating model output:
This method must be implemented in child classes

[Step 1: Duration 0.00 seconds]


حدث خطأ أثناء التشغيل: Error in generating model output:
This method must be implemented in child classes


In [8]:
!pip install litellm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.1/41.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.3/26.3 MB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 17.9 MB/s eta 0:00:00
  Attempting uninstall: importlib-metadata
    Found existing installation: importlib_metadata 9.0.0
    Uninstalling importlib_metadata-9.0.0:
      Successfully uninstalled importlib_metadata-9.0.0


In [9]:
# 1. تثبيت المكتبات (تأكد من تشغيل هذا السطر)
# !pip install llama-cpp-python smolagents duckduckgo-search

from smolagents import CodeAgent, DuckDuckGoSearchTool, Model
from smolagents.models import ChatMessage # ضروري جداً
from llama_cpp import Llama

# 2. بناء الكلاس بالمعايير الدقيقة للمكتبة
class ColabGGUFModel(Model):
    def __init__(self, model_path):
        super().__init__()
        print(f"جاري تحميل الموديل من: {model_path}")
        self.llm = Llama(
            model_path=model_path,
            n_ctx=2048,
            n_threads=2,
            n_gpu_layers=0
        )

    # التعديل الجوهري هنا: إضافة **kwargs لاستقبال أي متغيرات إضافية
    def forward(self, messages, stop_sequences=None, **kwargs) -> ChatMessage:
        # بناء الـ Prompt بصيغة Qwen
        prompt = ""
        for msg in messages:
            role = msg['role']
            content = msg['content']
            prompt += f"<|im_start|>{role}\n{content}<|im_end|>\n"
        prompt += "<|im_start|>assistant\n"

        # توليد الرد
        output = self.llm(
            prompt,
            max_tokens=512,
            stop=["<|im_end|>", "<|endoftext|>"],
            echo=False
        )

        generated_text = output['choices'][0]['text']

        # يجب إرجاع كائن ChatMessage وليس نصاً عادياً
        return ChatMessage(role="assistant", content=generated_text)

# 3. تشغيل النظام
model_path = "/content/Qwen2.5-1.5B-Instruct-Q4_K_M.gguf"

try:
    # إنشاء الموديل
    local_model = ColabGGUFModel(model_path)

    # إنشاء العميل مع أداة البحث
    agent = CodeAgent(
        tools=[DuckDuckGoSearchTool()],
        model=local_model,
    )

    print("\nبدأ التنفيذ... الموديل سيبدأ بكتابة كود البحث أولاً.")
    # تنفيذ السؤال
    result = agent.run("What is the current weather in Cairo?")

    print("\n" + "="*40)
    print(f"النتيجة النهائية: {result}")
    print("="*40)

except Exception as e:
    print(f"\nحدث خطأ أثناء التشغيل: {e}")

llama_model_loader: loaded meta data with 38 key-value pairs and 338 tensors from /content/Qwen2.5-1.5B-Instruct-Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen2
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Qwen2.5 1.5B Instruct
llama_model_loader: - kv   3:                           general.finetune str              = Instruct
llama_model_loader: - kv   4:                           general.basename str              = Qwen2.5
llama_model_loader: - kv   5:                         general.size_label str              = 1.5B
llama_model_loader: - kv   6:                            general.license str              = apache-2.0
llama_model_loader: - kv   7:          

جاري تحميل الموديل من: /content/Qwen2.5-1.5B-Instruct-Q4_K_M.gguf


llama_model_loader: - type  f32:  141 tensors
llama_model_loader: - type q4_K:  168 tensors
llama_model_loader: - type q6_K:   29 tensors
print_info: file format = GGUF V3 (latest)
print_info: file type   = Q4_K - Medium
print_info: file size   = 934.69 MiB (5.08 BPW) 
init_tokenizer: initializing tokenizer for type 2
load: 0 unused tokens
load: control token: 151660 '<|fim_middle|>' is not marked as EOG
load: control token: 151659 '<|fim_prefix|>' is not marked as EOG
load: control token: 151653 '<|vision_end|>' is not marked as EOG
load: control token: 151648 '<|box_start|>' is not marked as EOG
load: control token: 151646 '<|object_ref_start|>' is not marked as EOG
load: control token: 151649 '<|box_end|>' is not marked as EOG
load: control-looking token: 128247 '</s>' was not control-type; this is probably a bug in the model. its type will be overridden
load: control token: 151655 '<|image_pad|>' is not marked as EOG
load: control token: 151651 '<|quad_end|>' is not marked as EOG
l


بدأ التنفيذ... الموديل سيبدأ بكتابة كود البحث أولاً.


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ What is the current weather in Cairo?                                                                           │
│                                                                                                                 │
╰─ ColabGGUFModel - None ─────────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Error in generating model output:
This method must be implemented in child classes

[Step 1: Duration 0.01 seconds]


حدث خطأ أثناء التشغيل: Error in generating model output:
This method must be implemented in child classes


In [10]:
# 1. تنصيب المكتبات (تأكد من تنصيبها بعد الـ Restart)
# !pip install llama-cpp-python smolagents duckduckgo-search

import torch
from smolagents import CodeAgent, DuckDuckGoSearchTool, Model
from smolagents.models import ChatMessage
from llama_cpp import Llama
from typing import List, Dict, Optional

# 2. بناء الكلاس بطريقة تضمن تخطي خطأ NotImplementedError
class ColabGGUFModel(Model):
    def __init__(self, model_path: str):
        # لا نستدعي super().__init__() هنا لتجنب أي قيود من الكلاس الأصلي
        self.model_id = "qwen-local"
        print(f"جاري تحميل الموديل: {model_path}")
        self.llm = Llama(
            model_path=model_path,
            n_ctx=2048,
            n_threads=2,
            n_gpu_layers=0
        )

    # تنفيذ الدالة forward (المطلوبة من المكتبة)
    def forward(self, messages: List[Dict[str, str]], stop_sequences: Optional[List[str]] = None, **kwargs) -> ChatMessage:
        print(">>> الموديل بدأ المعالجة...")

        # بناء الـ Prompt
        prompt = ""
        for msg in messages:
            role = msg['role']
            content = msg['content']
            prompt += f"<|im_start|>{role}\n{content}<|im_end|>\n"
        prompt += "<|im_start|>assistant\n"

        # توليد النص
        output = self.llm(
            prompt,
            max_tokens=512,
            stop=["<|im_end|>", "<|endoftext|>"],
            echo=False
        )

        generated_text = output['choices'][0]['text']
        return ChatMessage(role="assistant", content=generated_text)

    # إضافة دالة __call__ لضمان عمل الوكيل في كل الحالات
    def __call__(self, messages, stop_sequences=None, **kwargs):
        return self.forward(messages, stop_sequences, **kwargs)

# 3. تشغيل النظام
model_path = "/content/Qwen2.5-1.5B-Instruct-Q4_K_M.gguf"

try:
    # إنشاء الموديل
    local_model = ColabGGUFModel(model_path)

    # التأكد يدوياً من وجود الدالة قبل تمريرها للوكيل
    if not hasattr(local_model, 'forward'):
        print("خطأ: الدالة forward غير موجودة!")

    # إنشاء العميل
    agent = CodeAgent(
        tools=[DuckDuckGoSearchTool()],
        model=local_model,
    )

    print("\n--- بدأ التنفيذ الفعلي ---")
    result = agent.run("ما هي عاصمة فرنسا؟")
    print(f"\nالنتيجة النهائية: {result}")

except Exception as e:
    print(f"\nحدث خطأ تقني: {e}")

llama_model_loader: loaded meta data with 38 key-value pairs and 338 tensors from /content/Qwen2.5-1.5B-Instruct-Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen2
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Qwen2.5 1.5B Instruct
llama_model_loader: - kv   3:                           general.finetune str              = Instruct
llama_model_loader: - kv   4:                           general.basename str              = Qwen2.5
llama_model_loader: - kv   5:                         general.size_label str              = 1.5B
llama_model_loader: - kv   6:                            general.license str              = apache-2.0
llama_model_loader: - kv   7:          

جاري تحميل الموديل: /content/Qwen2.5-1.5B-Instruct-Q4_K_M.gguf


init_tokenizer: initializing tokenizer for type 2
load: 0 unused tokens
load: control token: 151660 '<|fim_middle|>' is not marked as EOG
load: control token: 151659 '<|fim_prefix|>' is not marked as EOG
load: control token: 151653 '<|vision_end|>' is not marked as EOG
load: control token: 151648 '<|box_start|>' is not marked as EOG
load: control token: 151646 '<|object_ref_start|>' is not marked as EOG
load: control token: 151649 '<|box_end|>' is not marked as EOG
load: control-looking token: 128247 '</s>' was not control-type; this is probably a bug in the model. its type will be overridden
load: control token: 151655 '<|image_pad|>' is not marked as EOG
load: control token: 151651 '<|quad_end|>' is not marked as EOG
load: control token: 151647 '<|object_ref_end|>' is not marked as EOG
load: control token: 151652 '<|vision_start|>' is not marked as EOG
load: control token: 151654 '<|vision_pad|>' is not marked as EOG
load: control token: 151656 '<|video_pad|>' is not marked as EOG
lo


--- بدأ التنفيذ الفعلي ---


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ ما هي عاصمة فرنسا؟                                                                                              │
│                                                                                                                 │
╰─ ColabGGUFModel - qwen-local ───────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Error in generating model output:
This method must be implemented in child classes

[Step 1: Duration 0.00 seconds]


حدث خطأ تقني: Error in generating model output:
This method must be implemented in child classes


In [11]:
# 1. تنصيب المكتبات (تأكد من تنصيبها بعد الـ Restart)
# !pip install llama-cpp-python smolagents duckduckgo-search

import torch
from smolagents import CodeAgent, DuckDuckGoSearchTool, Model
from smolagents.models import ChatMessage
from llama_cpp import Llama
from typing import List, Dict, Optional

# 2. بناء الكلاس بطريقة تضمن تخطي خطأ NotImplementedError
class ColabGGUFModel(Model):
    def __init__(self, model_path: str):
        # لا نستدعي super().__init__() هنا لتجنب أي قيود من الكلاس الأصلي
        self.model_id = "qwen-local"
        print(f"جاري تحميل الموديل: {model_path}")
        self.llm = Llama(
            model_path=model_path,
            n_ctx=2048,
            n_threads=2,
            n_gpu_layers=0
        )

    # تنفيذ الدالة forward (المطلوبة من المكتبة)
    def forward(self, messages: List[Dict[str, str]], stop_sequences: Optional[List[str]] = None, **kwargs) -> ChatMessage:
        print(">>> الموديل بدأ المعالجة...")

        # بناء الـ Prompt
        prompt = ""
        for msg in messages:
            role = msg['role']
            content = msg['content']
            prompt += f"<|im_start|>{role}\n{content}<|im_end|>\n"
        prompt += "<|im_start|>assistant\n"

        # توليد النص
        output = self.llm(
            prompt,
            max_tokens=512,
            stop=["<|im_end|>", "<|endoftext|>"],
            echo=False
        )

        generated_text = output['choices'][0]['text']
        return ChatMessage(role="assistant", content=generated_text)

    # إضافة دالة __call__ لضمان عمل الوكيل في كل الحالات
    def __call__(self, messages, stop_sequences=None, **kwargs):
        return self.forward(messages, stop_sequences, **kwargs)

# 3. تشغيل النظام
model_path = "/content/Qwen2.5-1.5B-Instruct-Q4_K_M.gguf"

try:
    # إنشاء الموديل
    local_model = ColabGGUFModel(model_path)

    # التأكد يدوياً من وجود الدالة قبل تمريرها للوكيل
    if not hasattr(local_model, 'forward'):
        print("خطأ: الدالة forward غير موجودة!")

    # إنشاء العميل
    agent = CodeAgent(
        tools=[DuckDuckGoSearchTool()],
        model=local_model,
    )

    print("\n--- بدأ التنفيذ الفعلي ---")
    result = agent.run("ما هي عاصمة فرنسا؟")
    print(f"\nالنتيجة النهائية: {result}")

except Exception as e:
    print(f"\nحدث خطأ تقني: {e}")

llama_model_loader: loaded meta data with 38 key-value pairs and 338 tensors from /content/Qwen2.5-1.5B-Instruct-Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen2
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Qwen2.5 1.5B Instruct
llama_model_loader: - kv   3:                           general.finetune str              = Instruct
llama_model_loader: - kv   4:                           general.basename str              = Qwen2.5
llama_model_loader: - kv   5:                         general.size_label str              = 1.5B
llama_model_loader: - kv   6:                            general.license str              = apache-2.0
llama_model_loader: - kv   7:          

جاري تحميل الموديل: /content/Qwen2.5-1.5B-Instruct-Q4_K_M.gguf


init_tokenizer: initializing tokenizer for type 2
load: 0 unused tokens
load: control token: 151660 '<|fim_middle|>' is not marked as EOG
load: control token: 151659 '<|fim_prefix|>' is not marked as EOG
load: control token: 151653 '<|vision_end|>' is not marked as EOG
load: control token: 151648 '<|box_start|>' is not marked as EOG
load: control token: 151646 '<|object_ref_start|>' is not marked as EOG
load: control token: 151649 '<|box_end|>' is not marked as EOG
load: control-looking token: 128247 '</s>' was not control-type; this is probably a bug in the model. its type will be overridden
load: control token: 151655 '<|image_pad|>' is not marked as EOG
load: control token: 151651 '<|quad_end|>' is not marked as EOG
load: control token: 151647 '<|object_ref_end|>' is not marked as EOG
load: control token: 151652 '<|vision_start|>' is not marked as EOG
load: control token: 151654 '<|vision_pad|>' is not marked as EOG
load: control token: 151656 '<|video_pad|>' is not marked as EOG
lo


--- بدأ التنفيذ الفعلي ---


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ ما هي عاصمة فرنسا؟                                                                                              │
│                                                                                                                 │
╰─ ColabGGUFModel - qwen-local ───────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Error in generating model output:
This method must be implemented in child classes

[Step 1: Duration 0.00 seconds]


حدث خطأ تقني: Error in generating model output:
This method must be implemented in child classes


In [12]:
!wget https://huggingface.co/bartowski/smolagents_SmolVLM2-2.2B-Instruct-Agentic-GUI-GGUF/resolve/main/smolagents_SmolVLM2-2.2B-Instruct-Agentic-GUI-Q4_K_M.gguf

--2026-08-06 23:52:08--  https://huggingface.co/bartowski/smolagents_SmolVLM2-2.2B-Instruct-Agentic-GUI-GGUF/resolve/main/smolagents_SmolVLM2-2.2B-Instruct-Agentic-GUI-Q4_K_M.gguf
Resolving huggingface.co (huggingface.co)... 3.170.185.35, 3.170.185.33, 3.170.185.25, ...
Connecting to huggingface.co (huggingface.co)|3.170.185.35|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/68ed6ab551e3fa04b3ebce5e/516aaa562eb7984ea3ab4792915005bea60b6ac994d5b2958d2e0f2c33a21881?user_id=public&X-Xet-Cas-Uid=public&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27smolagents_SmolVLM2-2.2B-Instruct-Agentic-GUI-Q4_K_M.gguf%3B+filename%3D%22smolagents_SmolVLM2-2.2B-Instruct-Agentic-GUI-Q4_K_M.gguf%22%3B&Expires=1786063928&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjhlZDZhYjU1MWUzZmEwNGIzZWJjZTVlLzUxNmFhYTU2MmViNzk4NGVhM2FiNDc5MjkxNTAwNWJlYTYwYjZhYzk5NGQ1YjI5NThkMmUwZjJjMzN

In [13]:
from smolagents import CodeAgent, DuckDuckGoSearchTool, Model
from smolagents.models import ChatMessage
from llama_cpp import Llama
from typing import List, Dict, Optional

class SmolVLM2AgentModel(Model):
    def __init__(self, model_path: str):
        super().__init__()
        self.model_id = "smolvlm2-gguf"
        print(f"جاري تحميل الموديل الذكي: {model_path}")

        # إعدادات الموديل
        self.llm = Llama(
            model_path=model_path,
            n_ctx=4096,  # ذاكرة أكبر قليلاً لأنه موديل بصري
            n_threads=2,
            n_gpu_layers=0
        )

    def forward(self, messages: List[Dict[str, str]], stop_sequences: Optional[List[str]] = None, **kwargs) -> ChatMessage:
        # تحويل الرسائل لصيغة الموديل
        prompt = ""
        for msg in messages:
            role = msg['role']
            content = msg['content']
            # صيغة SmolVLM2 تشبه Qwen ولكنها أكثر دقة في الأدوات
            prompt += f"<|im_start|>{role}\n{content}<|im_end|>\n"
        prompt += "<|im_start|>assistant\n"

        # طلب توليد الكود أو الإجابة
        output = self.llm(
            prompt,
            max_tokens=1000, # زيادة التوكنز للسماح بكتابة كود مطول
            stop=["<|im_end|>", "<|endoftext|>"],
            echo=False
        )

        generated_text = output['choices'][0]['text']
        return ChatMessage(role="assistant", content=generated_text)

# مسار الموديل الجديد الذي حملته
model_path = "/content/smolagents_SmolVLM2-2.2B-Instruct-Agentic-GUI-Q4_K_M.gguf"

try:
    # 1. تهيئة الموديل
    local_model = SmolVLM2AgentModel(model_path)

    # 2. إنشاء العميل (الوكيل)
    # الموديل ده "عبقري" في استخدام الأدوات
    agent = CodeAgent(
        tools=[DuckDuckGoSearchTool()],
        model=local_model,
    )

    print("\n--- بدأ العميل العمل ---")
    # اطلب منه شيء يحتاج تفكير وكتابة كود
    result = agent.run("ابحث عن سعر البيتكوين اليوم وحوله للجنيه المصري")
    print(f"\nالنتيجة النهائية: {result}")

except Exception as e:
    print(f"\nحدث خطأ: {e}")

llama_model_loader: loaded meta data with 41 key-value pairs and 219 tensors from /content/smolagents_SmolVLM2-2.2B-Instruct-Agentic-GUI-Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = SmolVLM2 2.2B Instruct Agentic GUI
llama_model_loader: - kv   3:                           general.finetune str              = Instruct-Agentic-GUI
llama_model_loader: - kv   4:                           general.basename str              = SmolVLM2
llama_model_loader: - kv   5:                         general.size_label str              = 2.2B
llama_model_loader: - kv   6:                      general.dataset.count u32              = 

جاري تحميل الموديل الذكي: /content/smolagents_SmolVLM2-2.2B-Instruct-Agentic-GUI-Q4_K_M.gguf


load: control token:  49224 '<|reserved_special_token_33|>' is not marked as EOG
load: control token:  49223 '<|reserved_special_token_32|>' is not marked as EOG
load: control token:  49220 '<|reserved_special_token_29|>' is not marked as EOG
load: control token:  49218 '<|reserved_special_token_27|>' is not marked as EOG
load: control token:  49217 '<|reserved_special_token_26|>' is not marked as EOG
load: control token:  49215 '<|reserved_special_token_24|>' is not marked as EOG
load: control token:  49213 '<|reserved_special_token_22|>' is not marked as EOG
load: control token:  49212 '<|reserved_special_token_21|>' is not marked as EOG
load: control token:  49208 '<|reserved_special_token_17|>' is not marked as EOG
load: control token:  49204 '<|reserved_special_token_13|>' is not marked as EOG
load: control token:  49201 '<|reserved_special_token_10|>' is not marked as EOG
load: control token:  49200 '<|reserved_special_token_9|>' is not marked as EOG
load: control token:  49199 '


--- بدأ العميل العمل ---


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ ابحث عن سعر البيتكوين اليوم وحوله للجنيه المصري                                                                 │
│                                                                                                                 │
╰─ SmolVLM2AgentModel - smolvlm2-gguf ────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Error in generating model output:
This method must be implemented in child classes

[Step 1: Duration 0.01 seconds]


حدث خطأ: Error in generating model output:
This method must be implemented in child classes


In [15]:
# 1. تثبيت المكتبات (إذا لم تكن مثبتة في هذه الجلسة)
# !pip install llama-cpp-python smolagents duckduckgo-search

from smolagents import CodeAgent, DuckDuckGoSearchTool
from smolagents.models import ChatMessage
from llama_cpp import Llama
from typing import List, Dict, Optional

# 2. بناء كلاس "ذكي" لا يعتمد على الوراثة المباشرة المسببة للمشاكل
class SmolVLM2GGUFWrapper:
    def __init__(self, model_path: str):
        self.model_id = "SmolVLM2-Local"
        print(f"جاري تحميل الموديل: {model_path}")
        self.llm = Llama(
            model_path=model_path,
            n_ctx=4096,
            n_threads=2,
            n_gpu_layers=0 # اجعلها -1 إذا كنت تستخدم GPU في كولاب
        )

    # هذه الدالة هي ما يبحث عنه الوكيل (Agent)
    def __call__(self, messages: List[Dict[str, str]], stop_sequences: Optional[List[str]] = None, **kwargs) -> ChatMessage:
        print(">>> الموديل يفكر الآن...")

        # بناء الـ Prompt بصيغة SmolVLM2
        prompt = ""
        for msg in messages:
            role = msg['role']
            content = msg['content']
            prompt += f"<|im_start|>{role}\n{content}<|im_end|>\n"
        prompt += "<|im_start|>assistant\n"

        # توليد النص
        output = self.llm(
            prompt,
            max_tokens=1024,
            stop=["<|im_end|>", "<end_of_utterance>"],
            echo=False
        )

        # استخراج النص الناتج وتغليفه في ChatMessage
        generated_text = output['choices'][0]['text']
        return ChatMessage(role="assistant", content=generated_text)

# 3. تشغيل النظام
# تأكد أن هذا هو نفس المسار في اللوج الخاص بك
model_path = "/content/smolagents_SmolVLM2-2.2B-Instruct-Agentic-GUI-Q4_K_M.gguf"

try:
    # إنشاء الموديل باستخدام الـ Wrapper
    local_model = SmolVLM2GGUFWrapper(model_path)

    # إنشاء العميل (Agent)
    # ملاحظة: نمرر local_model مباشرة كـ model
    agent = CodeAgent(
        tools=[DuckDuckGoSearchTool()],
        model=local_model, # سيعمل لأننا أضفنا __call__
    )

    print("\n--- بدأ العميل العمل (المهامAgentic) ---")
    # طلب مهمة تتطلب بحث واستنتاج
    result = agent.run("What is the price of Bitcoin in USD and convert it to Egyptian Pounds?")

    print("\n" + "="*50)
    print(f"النتيجة النهائية:\n{result}")
    print("="*50)

except Exception as e:
    print(f"\nحدث خطأ تقني: {e}")

llama_model_loader: loaded meta data with 41 key-value pairs and 219 tensors from /content/smolagents_SmolVLM2-2.2B-Instruct-Agentic-GUI-Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = SmolVLM2 2.2B Instruct Agentic GUI
llama_model_loader: - kv   3:                           general.finetune str              = Instruct-Agentic-GUI
llama_model_loader: - kv   4:                           general.basename str              = SmolVLM2
llama_model_loader: - kv   5:                         general.size_label str              = 2.2B
llama_model_loader: - kv   6:                      general.dataset.count u32              = 

جاري تحميل الموديل: /content/smolagents_SmolVLM2-2.2B-Instruct-Agentic-GUI-Q4_K_M.gguf


load: control token:  49199 '<|reserved_special_token_8|>' is not marked as EOG
load: control token:  49197 '<|reserved_special_token_6|>' is not marked as EOG
load: control token:  49195 '<|reserved_special_token_4|>' is not marked as EOG
load: control token:  49194 '<|reserved_special_token_3|>' is not marked as EOG
load: control token:  49192 '<|reserved_special_token_1|>' is not marked as EOG
load: control token:  49191 '<|reserved_special_token_0|>' is not marked as EOG
load: control token:  49189 '<fake_token_around_image>' is not marked as EOG
load: control token:  49188 '<row_6_col_6>' is not marked as EOG
load: control token:  49186 '<row_6_col_4>' is not marked as EOG
load: control token:  49184 '<row_6_col_2>' is not marked as EOG
load: control token:  49183 '<row_6_col_1>' is not marked as EOG
load: control token:  49182 '<row_5_col_6>' is not marked as EOG
load: control token:  49181 '<row_5_col_5>' is not marked as EOG
load: control token:  49180 '<row_5_col_4>' is not ma


--- بدأ العميل العمل (المهامAgentic) ---


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ What is the price of Bitcoin in USD and convert it to Egyptian Pounds?                                          │
│                                                                                                                 │
╰─ SmolVLM2GGUFWrapper - SmolVLM2-Local ──────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Error in generating model output:
'SmolVLM2GGUFWrapper' object has no attribute 'generate'

[Step 1: Duration 0.01 seconds]


حدث خطأ تقني: Error in generating model output:
'SmolVLM2GGUFWrapper' object has no attribute 'generate'


In [16]:
from smolagents import CodeAgent, DuckDuckGoSearchTool, Model
from smolagents.models import ChatMessage
from llama_cpp import Llama
from typing import List, Dict, Optional

# بناء الكلاس مع الدالة generate التي تبحث عنها المكتبة
class SmolVLM2GGUFModel(Model):
    def __init__(self, model_path: str):
        super().__init__()
        self.model_id = "SmolVLM2-Local"
        print(f"جاري تحميل الموديل: {model_path}")

        self.llm = Llama(
            model_path=model_path,
            n_ctx=4096,
            n_threads=2,
            n_gpu_layers=0 # اجعلها -1 لو استخدمت GPU
        )

    # هذه هي الدالة التي يطلبها الوكيل (Agent) الآن
    def generate(
        self,
        messages: List[Dict[str, str]],
        stop_sequences: Optional[List[str]] = None,
        **kwargs
    ) -> ChatMessage:

        print(">>> الموديل يحلل المهمة الآن...")

        # بناء الـ Prompt
        prompt = ""
        for msg in messages:
            role = msg['role']
            content = msg['content']
            prompt += f"<|im_start|>{role}\n{content}<|im_end|>\n"
        prompt += "<|im_start|>assistant\n"

        # دمج Stop sequences الخاصة بالمكتبة مع الخاصة بالموديل
        stops = ["<|im_end|>", "<end_of_utterance>"]
        if stop_sequences:
            stops.extend(stop_sequences)

        # توليد النص
        output = self.llm(
            prompt,
            max_tokens=1024,
            stop=stops,
            echo=False,
            temperature=0.2 # درجة حرارة منخفضة لضمان دقة الكود البرمجي
        )

        generated_text = output['choices'][0]['text']

        # إرجاع النتيجة ككائن ChatMessage
        return ChatMessage(role="assistant", content=generated_text)

# مسار الموديل
model_path = "/content/smolagents_SmolVLM2-2.2B-Instruct-Agentic-GUI-Q4_K_M.gguf"

try:
    # 1. تهيئة الموديل
    local_model = SmolVLM2GGUFModel(model_path)

    # 2. إنشاء العميل
    agent = CodeAgent(
        tools=[DuckDuckGoSearchTool()],
        model=local_model,
        max_steps=5 # عدد الخطوات المسموح بها لحل المشكلة
    )

    print("\n--- بدأ العميل العمل ---")
    # طلب مهمة ذكية
    result = agent.run("What is the current price of Bitcoin in USD and convert it to Egyptian Pounds?")

    print("\n" + "="*50)
    print(f"النتيجة النهائية:\n{result}")
    print("="*50)

except Exception as e:
    print(f"\nحدث خطأ: {e}")

llama_model_loader: loaded meta data with 41 key-value pairs and 219 tensors from /content/smolagents_SmolVLM2-2.2B-Instruct-Agentic-GUI-Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = SmolVLM2 2.2B Instruct Agentic GUI
llama_model_loader: - kv   3:                           general.finetune str              = Instruct-Agentic-GUI
llama_model_loader: - kv   4:                           general.basename str              = SmolVLM2
llama_model_loader: - kv   5:                         general.size_label str              = 2.2B
llama_model_loader: - kv   6:                      general.dataset.count u32              = 

جاري تحميل الموديل: /content/smolagents_SmolVLM2-2.2B-Instruct-Agentic-GUI-Q4_K_M.gguf


load: control token:  49231 '<|reserved_special_token_40|>' is not marked as EOG
load: control token:  49229 '<|reserved_special_token_38|>' is not marked as EOG
load: control token:  49228 '<|reserved_special_token_37|>' is not marked as EOG
load: control token:  49227 '<|reserved_special_token_36|>' is not marked as EOG
load: control token:  49226 '<|reserved_special_token_35|>' is not marked as EOG
load: control token:  49225 '<|reserved_special_token_34|>' is not marked as EOG
load: control token:  49224 '<|reserved_special_token_33|>' is not marked as EOG
load: control token:  49223 '<|reserved_special_token_32|>' is not marked as EOG
load: control token:  49220 '<|reserved_special_token_29|>' is not marked as EOG
load: control token:  49218 '<|reserved_special_token_27|>' is not marked as EOG
load: control token:  49217 '<|reserved_special_token_26|>' is not marked as EOG
load: control token:  49215 '<|reserved_special_token_24|>' is not marked as EOG
load: control token:  49213 


--- بدأ العميل العمل ---


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ What is the current price of Bitcoin in USD and convert it to Egyptian Pounds?                                  │
│                                                                                                                 │
╰─ SmolVLM2GGUFModel - SmolVLM2-Local ────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

>>> الموديل يحلل المهمة الآن...


Error in generating model output:
'ChatMessage' object is not subscriptable

[Step 1: Duration 0.01 seconds]


حدث خطأ: Error in generating model output:
'ChatMessage' object is not subscriptable


In [17]:
from smolagents import CodeAgent, DuckDuckGoSearchTool, Model
from smolagents.models import ChatMessage
from llama_cpp import Llama
from typing import List, Dict, Optional, Any

class SmolVLM2GGUFModel(Model):
    def __init__(self, model_path: str):
        super().__init__()
        self.model_id = "SmolVLM2-Local"
        print(f"جاري تحميل الموديل الذكي: {model_path}")
        self.llm = Llama(
            model_path=model_path,
            n_ctx=4096,
            n_threads=2,
            n_gpu_layers=0
        )

    def generate(self, messages: List[Any], stop_sequences: Optional[List[str]] = None, **kwargs) -> ChatMessage:
        print(">>> الموديل يقوم بتحليل الرسائل وبناء خطة العمل...")

        # بناء الـ Prompt بطريقة صحيحة تتعامل مع كائنات ChatMessage
        prompt = ""
        for msg in messages:
            # الوصول للبيانات باستخدام . وليس [] لتجنب الخطأ
            role = msg.role
            content_raw = msg.content

            # معالجة المحتوى (لأن VLM قد يرسل قائمة بدلاً من نص)
            if isinstance(content_raw, list):
                text_content = ""
                for item in content_raw:
                    if isinstance(item, dict) and item.get("type") == "text":
                        text_content += item.get("text", "")
                content = text_content
            else:
                content = content_raw

            prompt += f"<|im_start|>{role}\n{content}<|im_end|>\n"

        prompt += "<|im_start|>assistant\n"

        # إعداد علامات التوقف
        stops = ["<|im_end|>", "<end_of_utterance>"]
        if stop_sequences:
            stops.extend(stop_sequences)

        # التوليد
        output = self.llm(
            prompt,
            max_tokens=1024,
            stop=stops,
            echo=False,
            temperature=0.1 # درجة حرارة منخفضة جداً لضمان دقة الكود
        )

        generated_text = output['choices'][0]['text']

        # إرجاع النتيجة ككائن ChatMessage
        return ChatMessage(role="assistant", content=generated_text)

# تشغيل النظام
model_path = "/content/smolagents_SmolVLM2-2.2B-Instruct-Agentic-GUI-Q4_K_M.gguf"

try:
    local_model = SmolVLM2GGUFModel(model_path)

    agent = CodeAgent(
        tools=[DuckDuckGoSearchTool()],
        model=local_model,
        max_steps=10 # زيادة الخطوات لأن الموديلات الصغيرة قد تحتاج محاولات أكثر
    )

    print("\n---بدأ الوكيل (Agent) في التفكير وكتابة الكود---")
    result = agent.run("Search for the current price of Bitcoin in USD and convert it to Egyptian Pounds.")

    print("\n" + "="*50)
    print(f"النتيجة النهائية:\n{result}")
    print("="*50)

except Exception as e:
    print(f"\nحدث خطأ: {e}")

llama_model_loader: loaded meta data with 41 key-value pairs and 219 tensors from /content/smolagents_SmolVLM2-2.2B-Instruct-Agentic-GUI-Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = SmolVLM2 2.2B Instruct Agentic GUI
llama_model_loader: - kv   3:                           general.finetune str              = Instruct-Agentic-GUI
llama_model_loader: - kv   4:                           general.basename str              = SmolVLM2
llama_model_loader: - kv   5:                         general.size_label str              = 2.2B
llama_model_loader: - kv   6:                      general.dataset.count u32              = 

جاري تحميل الموديل الذكي: /content/smolagents_SmolVLM2-2.2B-Instruct-Agentic-GUI-Q4_K_M.gguf


load: control token:  49201 '<|reserved_special_token_10|>' is not marked as EOG
load: control token:  49200 '<|reserved_special_token_9|>' is not marked as EOG
load: control token:  49199 '<|reserved_special_token_8|>' is not marked as EOG
load: control token:  49197 '<|reserved_special_token_6|>' is not marked as EOG
load: control token:  49195 '<|reserved_special_token_4|>' is not marked as EOG
load: control token:  49194 '<|reserved_special_token_3|>' is not marked as EOG
load: control token:  49192 '<|reserved_special_token_1|>' is not marked as EOG
load: control token:  49191 '<|reserved_special_token_0|>' is not marked as EOG
load: control token:  49189 '<fake_token_around_image>' is not marked as EOG
load: control token:  49188 '<row_6_col_6>' is not marked as EOG
load: control token:  49186 '<row_6_col_4>' is not marked as EOG
load: control token:  49184 '<row_6_col_2>' is not marked as EOG
load: control token:  49183 '<row_6_col_1>' is not marked as EOG
load: control token:  


---بدأ الوكيل (Agent) في التفكير وكتابة الكود---


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Search for the current price of Bitcoin in USD and convert it to Egyptian Pounds.                               │
│                                                                                                                 │
╰─ SmolVLM2GGUFModel - SmolVLM2-Local ────────────────────────────────────────────────────────────────────────────╯

...

llama_perf_context_print:        load time =  194352.53 ms
llama_perf_context_print: prompt eval time =   27401.01 ms /   281 tokens (   97.51 ms per token,    10.26 tokens per second)
llama_perf_context_print:        eval time =    7621.57 ms /    24 runs   (  317.57 ms per token,     3.15 tokens per second)
llama_perf_context_print:       total time =   35034.15 ms /   305 tokens
llama_perf_context_print:    graphs reused =         23


─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer('success')                                                                                          
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: success

[Step 2: Duration 35.07 seconds]


النتيجة النهائية:
success


In [ ]:

مبروك! لقد نجحت أخيراً في تشغيل الموديل والوكيل (Agent) معاً بدون أخطاء تقنية. ولكن، كما تلاحظ من النتيجة، هناك "مشكلة منطقية" حدثت، وإليك تفسيرها وكيفية حلها:
لماذا فشل الوكيل في جلب السعر؟ (تحليل اللوج)
هلوسة الـ GUI: الموديل الذي تستخدمه هو SmolVLM2-Agentic-GUI. هذا الموديل مُدرب خصيصاً للتعامل مع "واجهات المستخدم". لذلك، بدلاً من استخدام أداة البحث، تخيل الموديل أنه ينظر لشاشة وقام بكتابة أمر click(x=0.35, y=0.118) ليضغط على متصفح وهمي!
خطأ الأداة: الوكيل رفض تنفيذ الأمر لأن click ليست أداة مسموحة (أنت سمحت له بـ DuckDuckGoSearchTool فقط).
الاستسلام السريع: الموديل رد بكلمة success ليهرب من المهمة لأنه لم يعرف كيف يستخدم البحث النصي.
الحل: إجبار الموديل على استخدام البحث (التوجيه)
بما أن الموديل "بصري" ويميل لاستخدام أوامر الضغط (Click)، يجب أن نعدل "النظام" (System Prompt) لنخبره بوضوح: "لا تستخدم أوامر الشاشة، استخدم أداة البحث فقط".
قم بتعديل جزء تشغيل العميل في الكود الخاص بك إلى هذا الشكل:

In [18]:
# ... (نفس الكود السابق للكلاس SmolVLM2GGUFModel) ...

try:
    local_model = SmolVLM2GGUFModel(model_path)

    # إضافة تعليمات إضافية للوكيل (System Prompt)
    custom_instructions = """
    You are a helpful assistant.
    IMPORTANT: Do not try to use GUI tools like 'click' or 'type'.
    Only use the provided 'web_search' tool to find information.
    Output your reasoning and then the python code to call the tool.
    """

    agent = CodeAgent(
        tools=[DuckDuckGoSearchTool()],
        model=local_model,
        max_steps=5,
        additional_authorized_imports=["json"] # قد يحتاج الموديل لتحليل البيانات
    )

    print("\n--- بدأ الوكيل في العمل مع توجيهات صارمة ---")

    # الطلب مع التأكيد على استخدام الأداة
    query = "Use the search tool to find the current Bitcoin price in USD and then calculate its value in Egyptian Pounds (use 1 USD = 50 EGP for calculation)."

    result = agent.run(query, reset=True)

    print("\n" + "="*50)
    print(f"النتيجة النهائية:\n{result}")
    print("="*50)

except Exception as e:
    print(f"\nحدث خطأ: {e}")

llama_model_loader: loaded meta data with 41 key-value pairs and 219 tensors from /content/smolagents_SmolVLM2-2.2B-Instruct-Agentic-GUI-Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = SmolVLM2 2.2B Instruct Agentic GUI
llama_model_loader: - kv   3:                           general.finetune str              = Instruct-Agentic-GUI
llama_model_loader: - kv   4:                           general.basename str              = SmolVLM2
llama_model_loader: - kv   5:                         general.size_label str              = 2.2B
llama_model_loader: - kv   6:                      general.dataset.count u32              = 

جاري تحميل الموديل الذكي: /content/smolagents_SmolVLM2-2.2B-Instruct-Agentic-GUI-Q4_K_M.gguf


llama_model_loader: - kv  27:                  tokenizer.ggml.token_type arr[i32,49280]   = [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, ...
llama_model_loader: - kv  28:                      tokenizer.ggml.merges arr[str,48900]   = ["Ġ t", "Ġ a", "i n", "h e", "Ġ Ġ...
llama_model_loader: - kv  29:                tokenizer.ggml.bos_token_id u32              = 1
llama_model_loader: - kv  30:                tokenizer.ggml.eos_token_id u32              = 49279
llama_model_loader: - kv  31:            tokenizer.ggml.unknown_token_id u32              = 0
llama_model_loader: - kv  32:            tokenizer.ggml.padding_token_id u32              = 2
llama_model_loader: - kv  33:                    tokenizer.chat_template str              = <|im_start|>{% for message in message...
llama_model_loader: - kv  34:            tokenizer.ggml.add_space_prefix bool             = false
llama_model_loader: - kv  35:               general.quantization_version u32              = 2
llama_model_loader: - kv  36:    


--- بدأ الوكيل في العمل مع توجيهات صارمة ---


CPU : SSE3 = 1 | SSSE3 = 1 | AVX = 1 | AVX2 = 1 | F16C = 1 | FMA = 1 | BMI2 = 1 | LLAMAFILE = 1 | OPENMP = 1 | REPACK = 1 | 
Model metadata: {'quantize.imatrix.dataset': '/training_dir/calibration_datav5.txt', 'general.quantization_version': '2', 'tokenizer.chat_template': "<|im_start|>{% for message in messages %}{{message['role'] | capitalize}}{% if message['content'][0]['type'] == 'image' %}{{':'}}{% else %}{{': '}}{% endif %}{% for line in message['content'] %}{% if line['type'] == 'text' %}{{line['text']}}{% elif line['type'] == 'image' %}{{ '<image>' }}{% endif %}{% endfor %}<end_of_utterance>\n{% endfor %}{% if add_generation_prompt %}{{ 'Assistant:' }}{% endif %}", 'general.dataset.0.version': '2', 'quantize.imatrix.entries_count': '168', 'llama.attention.head_count_kv': '32', 'general.dataset.0.name': 'Aguvis Stage 2', 'general.dataset.count': '1', 'general.size_label': '2.2B', 'general.type': 'model', 'general.file_type': '15', 'general.dataset.0.organization': 'Smolagents', 

...

llama_perf_context_print:        load time =  196161.76 ms
llama_perf_context_print: prompt eval time =   26171.76 ms /   267 tokens (   98.02 ms per token,    10.20 tokens per second)
llama_perf_context_print:        eval time =    5664.79 ms /    24 runs   (  236.03 ms per token,     4.24 tokens per second)
llama_perf_context_print:       total time =   31847.31 ms /   291 tokens
llama_perf_context_print:    graphs reused =         23


─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer('success')                                                                                          
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: success

[Step 2: Duration 31.87 seconds]


النتيجة النهائية:
success


In [19]:
from smolagents import CodeAgent, DuckDuckGoSearchTool, Model
from smolagents.models import ChatMessage
from llama_cpp import Llama
from typing import List, Dict, Optional, Any

# بناء الكلاس لـ Qwen2.5 مع دالة generate المطلوبة
class QwenGGUFModel(Model):
    def __init__(self, model_path: str):
        super().__init__()
        self.model_id = "Qwen2.5-Local"
        print(f"جاري تحميل Qwen من المسار: {model_path}")
        self.llm = Llama(
            model_path=model_path,
            n_ctx=4096,
            n_threads=2,
            n_gpu_layers=0
        )

    def generate(self, messages: List[Any], stop_sequences: Optional[List[str]] = None, **kwargs) -> ChatMessage:
        # بناء الـ Prompt بصيغة Qwen
        prompt = ""
        for msg in messages:
            role = msg.role
            content_raw = msg.content
            # استخلاص النص فقط
            if isinstance(content_raw, list):
                content = "".join([item.get("text", "") for item in content_raw if item.get("type") == "text"])
            else:
                content = content_raw

            prompt += f"<|im_start|>{role}\n{content}<|im_end|>\n"

        prompt += "<|im_start|>assistant\n"

        # إعداد علامات التوقف
        stops = ["<|im_end|>", "<|endoftext|>"]
        if stop_sequences:
            stops.extend(stop_sequences)

        # التوليد
        output = self.llm(
            prompt,
            max_tokens=1024,
            stop=stops,
            echo=False,
            temperature=0.1
        )

        return ChatMessage(role="assistant", content=output['choices'][0]['text'])

# مسار موديل Qwen الذي تملكه
model_path = "/content/Qwen2.5-1.5B-Instruct-Q4_K_M.gguf"

try:
    # 1. تهيئة الموديل
    local_model = QwenGGUFModel(model_path)

    # 2. إنشاء الوكيل
    agent = CodeAgent(
        tools=[DuckDuckGoSearchTool()],
        model=local_model,
        max_steps=5
    )

    print("\n--- بدأ Qwen في البحث والحساب ---")
    # طلب المهمة
    result = agent.run("Search for the current Bitcoin price in USD and convert it to Egyptian Pounds (assume 1 USD = 50 EGP).")

    print("\n" + "="*50)
    print(f"النتيجة النهائية:\n{result}")
    print("="*50)

except Exception as e:
    print(f"\nحدث خطأ: {e}")

llama_model_loader: loaded meta data with 38 key-value pairs and 338 tensors from /content/Qwen2.5-1.5B-Instruct-Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen2
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Qwen2.5 1.5B Instruct
llama_model_loader: - kv   3:                           general.finetune str              = Instruct
llama_model_loader: - kv   4:                           general.basename str              = Qwen2.5
llama_model_loader: - kv   5:                         general.size_label str              = 1.5B
llama_model_loader: - kv   6:                            general.license str              = apache-2.0
llama_model_loader: - kv   7:          

جاري تحميل Qwen من المسار: /content/Qwen2.5-1.5B-Instruct-Q4_K_M.gguf


init_tokenizer: initializing tokenizer for type 2
load: 0 unused tokens
load: control token: 151660 '<|fim_middle|>' is not marked as EOG
load: control token: 151659 '<|fim_prefix|>' is not marked as EOG
load: control token: 151653 '<|vision_end|>' is not marked as EOG
load: control token: 151648 '<|box_start|>' is not marked as EOG
load: control token: 151646 '<|object_ref_start|>' is not marked as EOG
load: control token: 151649 '<|box_end|>' is not marked as EOG
load: control-looking token: 128247 '</s>' was not control-type; this is probably a bug in the model. its type will be overridden
load: control token: 151655 '<|image_pad|>' is not marked as EOG
load: control token: 151651 '<|quad_end|>' is not marked as EOG
load: control token: 151647 '<|object_ref_end|>' is not marked as EOG
load: control token: 151652 '<|vision_start|>' is not marked as EOG
load: control token: 151654 '<|vision_pad|>' is not marked as EOG
load: control token: 151656 '<|video_pad|>' is not marked as EOG
lo


--- بدأ Qwen في البحث والحساب ---


CPU : SSE3 = 1 | SSSE3 = 1 | AVX = 1 | AVX2 = 1 | F16C = 1 | FMA = 1 | BMI2 = 1 | LLAMAFILE = 1 | OPENMP = 1 | REPACK = 1 | 
Model metadata: {'quantize.imatrix.chunks_count': '128', 'quantize.imatrix.entries_count': '196', 'quantize.imatrix.file': '/models_out/Qwen2.5-1.5B-Instruct-GGUF/Qwen2.5-1.5B-Instruct.imatrix', 'general.base_model.0.repo_url': 'https://huggingface.co/Qwen/Qwen2.5-1.5B', 'general.license': 'apache-2.0', 'qwen2.attention.head_count_kv': '2', 'tokenizer.chat_template': '{%- if tools %}\n    {{- \'<|im_start|>system\\n\' }}\n    {%- if messages[0][\'role\'] == \'system\' %}\n        {{- messages[0][\'content\'] }}\n    {%- else %}\n        {{- \'You are Qwen, created by Alibaba Cloud. You are a helpful assistant.\' }}\n    {%- endif %}\n    {{- "\\n\\n# Tools\\n\\nYou may call one or more functions to assist with the user query.\\n\\nYou are provided with function signatures within <tools></tools> XML tags:\\n<tools>" }}\n    {%- for tool in tools %}\n        {{- "\

...

[Step 2: Duration 116.31 seconds]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 3 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Error in generating model output:
Requested tokens (4678) exceed context window of 4096

[Step 3: Duration 0.03 seconds]


حدث خطأ: Error in generating model output:
Requested tokens (4678) exceed context window of 4096


n_ctx=8192,

In [1]:
from smolagents import CodeAgent, DuckDuckGoSearchTool, Model
from smolagents.models import ChatMessage
from llama_cpp import Llama
from typing import List, Dict, Optional, Any

# بناء الكلاس لـ Qwen2.5 مع دالة generate المطلوبة
class QwenGGUFModel(Model):
    def __init__(self, model_path: str):
        super().__init__()
        self.model_id = "Qwen2.5-Local"
        print(f"جاري تحميل Qwen من المسار: {model_path}")
        self.llm = Llama(
            model_path=model_path,
            n_ctx=8192,
            n_threads=2,
            n_gpu_layers=0
        )

    def generate(self, messages: List[Any], stop_sequences: Optional[List[str]] = None, **kwargs) -> ChatMessage:
        # بناء الـ Prompt بصيغة Qwen
        prompt = ""
        for msg in messages:
            role = msg.role
            content_raw = msg.content
            # استخلاص النص فقط
            if isinstance(content_raw, list):
                content = "".join([item.get("text", "") for item in content_raw if item.get("type") == "text"])
            else:
                content = content_raw

            prompt += f"<|im_start|>{role}\n{content}<|im_end|>\n"

        prompt += "<|im_start|>assistant\n"

        # إعداد علامات التوقف
        stops = ["<|im_end|>", "<|endoftext|>"]
        if stop_sequences:
            stops.extend(stop_sequences)

        # التوليد
        output = self.llm(
            prompt,
            max_tokens=1024,
            stop=stops,
            echo=False,
            temperature=0.1
        )

        return ChatMessage(role="assistant", content=output['choices'][0]['text'])

# مسار موديل Qwen الذي تملكه
model_path = "/content/Qwen2.5-1.5B-Instruct-Q4_K_M.gguf"

try:
    # 1. تهيئة الموديل
    local_model = QwenGGUFModel(model_path)

    # 2. إنشاء الوكيل
    agent = CodeAgent(
        tools=[DuckDuckGoSearchTool()],
        model=local_model,
        max_steps=5
    )

    print("\n--- بدأ Qwen في البحث والحساب ---")
    # طلب المهمة
    result = agent.run("Search for the current Bitcoin price in USD and convert it to Egyptian Pounds (assume 1 USD = 50 EGP).")

    print("\n" + "="*50)
    print(f"النتيجة النهائية:\n{result}")
    print("="*50)

except Exception as e:
    print(f"\nحدث خطأ: {e}")

llama_model_loader: loaded meta data with 38 key-value pairs and 338 tensors from /content/Qwen2.5-1.5B-Instruct-Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen2
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Qwen2.5 1.5B Instruct
llama_model_loader: - kv   3:                           general.finetune str              = Instruct
llama_model_loader: - kv   4:                           general.basename str              = Qwen2.5
llama_model_loader: - kv   5:                         general.size_label str              = 1.5B
llama_model_loader: - kv   6:                            general.license str              = apache-2.0
llama_model_loader: - kv   7:          

جاري تحميل Qwen من المسار: /content/Qwen2.5-1.5B-Instruct-Q4_K_M.gguf


llama_model_loader: - kv  25:                      tokenizer.ggml.tokens arr[str,151936]  = ["!", "\"", "#", "$", "%", "&", "'", ...
llama_model_loader: - kv  26:                  tokenizer.ggml.token_type arr[i32,151936]  = [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...
llama_model_loader: - kv  27:                      tokenizer.ggml.merges arr[str,151387]  = ["Ġ Ġ", "ĠĠ ĠĠ", "i n", "Ġ t",...
llama_model_loader: - kv  28:                tokenizer.ggml.eos_token_id u32              = 151645
llama_model_loader: - kv  29:            tokenizer.ggml.padding_token_id u32              = 151643
llama_model_loader: - kv  30:                tokenizer.ggml.bos_token_id u32              = 151643
llama_model_loader: - kv  31:               tokenizer.ggml.add_bos_token bool             = false
llama_model_loader: - kv  32:                    tokenizer.chat_template str              = {%- if tools %}\n    {{- '<|im_start|>...
llama_model_loader: - kv  33:               general.quantization_version u32   


--- بدأ Qwen في البحث والحساب ---


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Search for the current Bitcoin price in USD and convert it to Egyptian Pounds (assume 1 USD = 50 EGP).          │
│                                                                                                                 │
╰─ QwenGGUFModel - Qwen2.5-Local ─────────────────────────────────────────────────────────────────────────────────╯

...

[Step 5: Duration 118.48 seconds]

Llama.generate: 5 prefix-match hit, remaining 4019 prompt tokens to eval
llama_perf_context_print:        load time =  135843.17 ms
llama_perf_context_print: prompt eval time =  288225.78 ms /  4019 tokens (   71.72 ms per token,    13.94 tokens per second)
llama_perf_context_print:        eval time =   25016.93 ms /   129 runs   (  193.93 ms per token,     5.16 tokens per second)
llama_perf_context_print:       total time =  313385.50 ms /  4148 tokens
llama_perf_context_print:    graphs reused =        127


Reached max steps.

[Step 6: Duration 313.41 seconds]


النتيجة النهائية:
Thought: I need to search for the current Bitcoin price in USD and then convert it to Egyptian Pounds. I will use the 'web_search' tool to find the current Bitcoin price in USD and then use the 'currency_converter' tool to convert it to Egyptian Pounds.
<code>
bitcoin_price_usd = web_search(query="current Bitcoin price in USD")
print("Current Bitcoin price in USD:", bitcoin_price_usd)
bitcoin_price_egp = currency_converter(amount=bitcoin_price_usd, from_currency="USD", to_currency="EGP")
print("Current Bitcoin price in EGP:", bitcoin_price_egp)
</code>


 llama_model_loader: loaded meta data with 38 key-value pairs and 338 tensors from /content/Qwen2.5-1.5B-Instruct-Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen2
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Qwen2.5 1.5B Instruct
llama_model_loader: - kv   3:                           general.finetune str              = Instruct
llama_model_loader: - kv   4:                           general.basename str              = Qwen2.5
llama_model_loader: - kv   5:                         general.size_label str              = 1.5B
llama_model_loader: - kv   6:                            general.license str              = apache-2.0
llama_model_loader: - kv   7:                       general.license.link str              = https://huggingface.co/Qwen/Qwen2.5-1...
llama_model_loader: - kv   8:                   general.base_model.count u32              = 1
llama_model_loader: - kv   9:                  general.base_model.0.name str              = Qwen2.5 1.5B
llama_model_loader: - kv  10:          general.base_model.0.organization str              = Qwen
llama_model_loader: - kv  11:              general.base_model.0.repo_url str              = https://huggingface.co/Qwen/Qwen2.5-1.5B
llama_model_loader: - kv  12:                               general.tags arr[str,2]       = ["chat", "text-generation"]
llama_model_loader: - kv  13:                          general.languages arr[str,1]       = ["en"]
llama_model_loader: - kv  14:                          qwen2.block_count u32              = 28
llama_model_loader: - kv  15:                       qwen2.context_length u32              = 32768
llama_model_loader: - kv  16:                     qwen2.embedding_length u32              = 1536
llama_model_loader: - kv  17:                  qwen2.feed_forward_length u32              = 8960
llama_model_loader: - kv  18:                 qwen2.attention.head_count u32              = 12
llama_model_loader: - kv  19:              qwen2.attention.head_count_kv u32              = 2
llama_model_loader: - kv  20:                       qwen2.rope.freq_base f32              = 1000000.000000
llama_model_loader: - kv  21:     qwen2.attention.layer_norm_rms_epsilon f32              = 0.000001
llama_model_loader: - kv  22:                          general.file_type u32              = 15
llama_model_loader: - kv  23:                       tokenizer.ggml.model str              = gpt2
llama_model_loader: - kv  24:                         tokenizer.ggml.pre str              = qwen2
جاري تحميل Qwen من المسار: /content/Qwen2.5-1.5B-Instruct-Q4_K_M.gguf
llama_model_loader: - kv  25:                      tokenizer.ggml.tokens arr[str,151936]  = ["!", "\"", "#", "$", "%", "&", "'", ...
llama_model_loader: - kv  26:                  tokenizer.ggml.token_type arr[i32,151936]  = [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...
llama_model_loader: - kv  27:                      tokenizer.ggml.merges arr[str,151387]  = ["Ġ Ġ", "ĠĠ ĠĠ", "i n", "Ġ t",...
llama_model_loader: - kv  28:                tokenizer.ggml.eos_token_id u32              = 151645
llama_model_loader: - kv  29:            tokenizer.ggml.padding_token_id u32              = 151643
llama_model_loader: - kv  30:                tokenizer.ggml.bos_token_id u32              = 151643
llama_model_loader: - kv  31:               tokenizer.ggml.add_bos_token bool             = false
llama_model_loader: - kv  32:                    tokenizer.chat_template str              = {%- if tools %}\n    {{- '<|im_start|>...
llama_model_loader: - kv  33:               general.quantization_version u32              = 2
llama_model_loader: - kv  34:                      quantize.imatrix.file str              = /models_out/Qwen2.5-1.5B-Instruct-GGU...
llama_model_loader: - kv  35:                   quantize.imatrix.dataset str              = /training_dir/calibration_datav3.txt
llama_model_loader: - kv  36:             quantize.imatrix.entries_count i32              = 196
llama_model_loader: - kv  37:              quantize.imatrix.chunks_count i32              = 128
llama_model_loader: - type  f32:  141 tensors
llama_model_loader: - type q4_K:  168 tensors
llama_model_loader: - type q6_K:   29 tensors
print_info: file format = GGUF V3 (latest)
print_info: file type   = Q4_K - Medium
print_info: file size   = 934.69 MiB (5.08 BPW)
init_tokenizer: initializing tokenizer for type 2
load: 0 unused tokens
load: control token: 151660 '<|fim_middle|>' is not marked as EOG
load: control token: 151659 '<|fim_prefix|>' is not marked as EOG
load: control token: 151653 '<|vision_end|>' is not marked as EOG
load: control token: 151648 '<|box_start|>' is not marked as EOG
load: control token: 151646 '<|object_ref_start|>' is not marked as EOG
load: control token: 151649 '<|box_end|>' is not marked as EOG
load: control-looking token: 128247 '</s>' was not control-type; this is probably a bug in the model. its type will be overridden
load: control token: 151655 '<|image_pad|>' is not marked as EOG
load: control token: 151651 '<|quad_end|>' is not marked as EOG
load: control token: 151647 '<|object_ref_end|>' is not marked as EOG
load: control token: 151652 '<|vision_start|>' is not marked as EOG
load: control token: 151654 '<|vision_pad|>' is not marked as EOG
load: control token: 151656 '<|video_pad|>' is not marked as EOG
load: control token: 151644 '<|im_start|>' is not marked as EOG
load: control token: 151661 '<|fim_suffix|>' is not marked as EOG
load: control token: 151650 '<|quad_start|>' is not marked as EOG
load: printing all EOG tokens:
load:   - 128247 ('</s>')
load:   - 151643 ('<|endoftext|>')
load:   - 151645 ('<|im_end|>')
load:   - 151662 ('<|fim_pad|>')
load:   - 151663 ('<|repo_name|>')
load:   - 151664 ('<|file_sep|>')
load: special tokens cache size = 23
load: token to piece cache size = 0.9310 MB
print_info: arch                  = qwen2
print_info: vocab_only            = 0
print_info: no_alloc              = 0
print_info: n_ctx_train           = 32768
print_info: n_embd_inp            = 1536
print_info: n_embd                = 1536
print_info: n_embd_out            = 1536
print_info: n_layer               = 28
print_info: n_layer_all           = 28
print_info: n_head                = 12
print_info: n_head_kv             = 2
print_info: n_rot                 = 128
print_info: n_swa                 = 0
print_info: is_swa_any            = 0
print_info: n_embd_head_k         = 128
print_info: n_embd_head_v         = 128
print_info: n_gqa                 = 6
print_info: n_embd_k_gqa          = 256
print_info: n_embd_v_gqa          = 256
print_info: f_norm_eps            = 0.0e+00
print_info: f_norm_rms_eps        = 1.0e-06
print_info: f_clamp_kqv           = 0.0e+00
print_info: f_max_alibi_bias      = 0.0e+00
print_info: f_logit_scale         = 0.0e+00
print_info: f_attn_scale          = 0.0e+00
print_info: f_attn_value_scale    = 0.0000
print_info: n_ff                  = 8960
print_info: n_expert              = 0
print_info: n_expert_used         = 0
print_info: n_expert_groups       = 0
print_info: n_group_used          = 0
print_info: causal attn           = 1
print_info: pooling type          = -1
print_info: rope type             = 2
print_info: rope scaling          = linear
print_info: freq_base_train       = 1000000.0
print_info: freq_scale_train      = 1
print_info: n_ctx_orig_yarn       = 32768
print_info: rope_yarn_log_mul     = 0.0000
print_info: rope_finetuned        = unknown
print_info: model type            = 1.5B
print_info: model params          = 1.54 B
print_info: general.name          = Qwen2.5 1.5B Instruct
print_info: vocab type            = BPE
print_info: n_vocab               = 151936
print_info: n_merges              = 151387
print_info: BOS token             = 151643 '<|endoftext|>'
print_info: EOS token             = 151645 '<|im_end|>'
print_info: EOT token             = 151645 '<|im_end|>'
print_info: PAD token             = 151643 '<|endoftext|>'
print_info: LF token              = 198 'Ċ'
print_info: FIM PRE token         = 151659 '<|fim_prefix|>'
print_info: FIM SUF token         = 151661 '<|fim_suffix|>'
print_info: FIM MID token         = 151660 '<|fim_middle|>'
print_info: FIM PAD token         = 151662 '<|fim_pad|>'
print_info: FIM REP token         = 151663 '<|repo_name|>'
print_info: FIM SEP token         = 151664 '<|file_sep|>'
print_info: EOG token             = 128247 '</s>'
print_info: EOG token             = 151643 '<|endoftext|>'
print_info: EOG token             = 151645 '<|im_end|>'
print_info: EOG token             = 151662 '<|fim_pad|>'
print_info: EOG token             = 151663 '<|repo_name|>'
print_info: EOG token             = 151664 '<|file_sep|>'
print_info: max token length      = 256
load_tensors: loading model tensors, this can take a while... (mmap = true, direct_io = false)
load_tensors: layer   0 assigned to device CPU, is_swa = 0
load_tensors: layer   1 assigned to device CPU, is_swa = 0
load_tensors: layer   2 assigned to device CPU, is_swa = 0
load_tensors: layer   3 assigned to device CPU, is_swa = 0
load_tensors: layer   4 assigned to device CPU, is_swa = 0
load_tensors: layer   5 assigned to device CPU, is_swa = 0
load_tensors: layer   6 assigned to device CPU, is_swa = 0
load_tensors: layer   7 assigned to device CPU, is_swa = 0
load_tensors: layer   8 assigned to device CPU, is_swa = 0
load_tensors: layer   9 assigned to device CPU, is_swa = 0
load_tensors: layer  10 assigned to device CPU, is_swa = 0
load_tensors: layer  11 assigned to device CPU, is_swa = 0
load_tensors: layer  12 assigned to device CPU, is_swa = 0
load_tensors: layer  13 assigned to device CPU, is_swa = 0
load_tensors: layer  14 assigned to device CPU, is_swa = 0
load_tensors: layer  15 assigned to device CPU, is_swa = 0
load_tensors: layer  16 assigned to device CPU, is_swa = 0
load_tensors: layer  17 assigned to device CPU, is_swa = 0
load_tensors: layer  18 assigned to device CPU, is_swa = 0
load_tensors: layer  19 assigned to device CPU, is_swa = 0
load_tensors: layer  20 assigned to device CPU, is_swa = 0
load_tensors: layer  21 assigned to device CPU, is_swa = 0
load_tensors: layer  22 assigned to device CPU, is_swa = 0
load_tensors: layer  23 assigned to device CPU, is_swa = 0
load_tensors: layer  24 assigned to device CPU, is_swa = 0
load_tensors: layer  25 assigned to device CPU, is_swa = 0
load_tensors: layer  26 assigned to device CPU, is_swa = 0
load_tensors: layer  27 assigned to device CPU, is_swa = 0
load_tensors: layer  28 assigned to device CPU, is_swa = 0
create_tensor: loading tensor token_embd.weight
create_tensor: loading tensor output_norm.weight
create_tensor: loading tensor blk.0.attn_norm.weight
create_tensor: loading tensor blk.0.attn_q.weight
create_tensor: loading tensor blk.0.attn_k.weight
create_tensor: loading tensor blk.0.attn_v.weight
create_tensor: loading tensor blk.0.attn_q.bias
create_tensor: loading tensor blk.0.attn_k.bias
create_tensor: loading tensor blk.0.attn_v.bias
create_tensor: loading tensor blk.0.attn_output.weight
create_tensor: loading tensor blk.0.ffn_norm.weight
create_tensor: loading tensor blk.0.ffn_gate.weight
create_tensor: loading tensor blk.0.ffn_down.weight
create_tensor: loading tensor blk.0.ffn_up.weight
create_tensor: loading tensor blk.1.attn_norm.weight
create_tensor: loading tensor blk.1.attn_q.weight
create_tensor: loading tensor blk.1.attn_k.weight
create_tensor: loading tensor blk.1.attn_v.weight
create_tensor: loading tensor blk.1.attn_q.bias
create_tensor: loading tensor blk.1.attn_k.bias
create_tensor: loading tensor blk.1.attn_v.bias
create_tensor: loading tensor blk.1.attn_output.weight
create_tensor: loading tensor blk.1.ffn_norm.weight
create_tensor: loading tensor blk.1.ffn_gate.weight
create_tensor: loading tensor blk.1.ffn_down.weight
create_tensor: loading tensor blk.1.ffn_up.weight
create_tensor: loading tensor blk.2.attn_norm.weight
create_tensor: loading tensor blk.2.attn_q.weight
create_tensor: loading tensor blk.2.attn_k.weight
create_tensor: loading tensor blk.2.attn_v.weight
create_tensor: loading tensor blk.2.attn_q.bias
create_tensor: loading tensor blk.2.attn_k.bias
create_tensor: loading tensor blk.2.attn_v.bias
create_tensor: loading tensor blk.2.attn_output.weight
create_tensor: loading tensor blk.2.ffn_norm.weight
create_tensor: loading tensor blk.2.ffn_gate.weight
create_tensor: loading tensor blk.2.ffn_down.weight
create_tensor: loading tensor blk.2.ffn_up.weight
create_tensor: loading tensor blk.3.attn_norm.weight
create_tensor: loading tensor blk.3.attn_q.weight
create_tensor: loading tensor blk.3.attn_k.weight
create_tensor: loading tensor blk.3.attn_v.weight
create_tensor: loading tensor blk.3.attn_q.bias
create_tensor: loading tensor blk.3.attn_k.bias
create_tensor: loading tensor blk.3.attn_v.bias
create_tensor: loading tensor blk.3.attn_output.weight
create_tensor: loading tensor blk.3.ffn_norm.weight
create_tensor: loading tensor blk.3.ffn_gate.weight
create_tensor: loading tensor blk.3.ffn_down.weight
create_tensor: loading tensor blk.3.ffn_up.weight
create_tensor: loading tensor blk.4.attn_norm.weight
create_tensor: loading tensor blk.4.attn_q.weight
create_tensor: loading tensor blk.4.attn_k.weight
create_tensor: loading tensor blk.4.attn_v.weight
create_tensor: loading tensor blk.4.attn_q.bias
create_tensor: loading tensor blk.4.attn_k.bias
create_tensor: loading tensor blk.4.attn_v.bias
create_tensor: loading tensor blk.4.attn_output.weight
create_tensor: loading tensor blk.4.ffn_norm.weight
create_tensor: loading tensor blk.4.ffn_gate.weight
create_tensor: loading tensor blk.4.ffn_down.weight
create_tensor: loading tensor blk.4.ffn_up.weight
create_tensor: loading tensor blk.5.attn_norm.weight
create_tensor: loading tensor blk.5.attn_q.weight
create_tensor: loading tensor blk.5.attn_k.weight
create_tensor: loading tensor blk.5.attn_v.weight
create_tensor: loading tensor blk.5.attn_q.bias
create_tensor: loading tensor blk.5.attn_k.bias
create_tensor: loading tensor blk.5.attn_v.bias
create_tensor: loading tensor blk.5.attn_output.weight
create_tensor: loading tensor blk.5.ffn_norm.weight
create_tensor: loading tensor blk.5.ffn_gate.weight
create_tensor: loading tensor blk.5.ffn_down.weight
create_tensor: loading tensor blk.5.ffn_up.weight
create_tensor: loading tensor blk.6.attn_norm.weight
create_tensor: loading tensor blk.6.attn_q.weight
create_tensor: loading tensor blk.6.attn_k.weight
create_tensor: loading tensor blk.6.attn_v.weight
create_tensor: loading tensor blk.6.attn_q.bias
create_tensor: loading tensor blk.6.attn_k.bias
create_tensor: loading tensor blk.6.attn_v.bias
create_tensor: loading tensor blk.6.attn_output.weight
create_tensor: loading tensor blk.6.ffn_norm.weight
create_tensor: loading tensor blk.6.ffn_gate.weight
create_tensor: loading tensor blk.6.ffn_down.weight
create_tensor: loading tensor blk.6.ffn_up.weight
create_tensor: loading tensor blk.7.attn_norm.weight
create_tensor: loading tensor blk.7.attn_q.weight
create_tensor: loading tensor blk.7.attn_k.weight
create_tensor: loading tensor blk.7.attn_v.weight
create_tensor: loading tensor blk.7.attn_q.bias
create_tensor: loading tensor blk.7.attn_k.bias
create_tensor: loading tensor blk.7.attn_v.bias
create_tensor: loading tensor blk.7.attn_output.weight
create_tensor: loading tensor blk.7.ffn_norm.weight
create_tensor: loading tensor blk.7.ffn_gate.weight
create_tensor: loading tensor blk.7.ffn_down.weight
create_tensor: loading tensor blk.7.ffn_up.weight
create_tensor: loading tensor blk.8.attn_norm.weight
create_tensor: loading tensor blk.8.attn_q.weight
create_tensor: loading tensor blk.8.attn_k.weight
create_tensor: loading tensor blk.8.attn_v.weight
create_tensor: loading tensor blk.8.attn_q.bias
create_tensor: loading tensor blk.8.attn_k.bias
create_tensor: loading tensor blk.8.attn_v.bias
create_tensor: loading tensor blk.8.attn_output.weight
create_tensor: loading tensor blk.8.ffn_norm.weight
create_tensor: loading tensor blk.8.ffn_gate.weight
create_tensor: loading tensor blk.8.ffn_down.weight
create_tensor: loading tensor blk.8.ffn_up.weight
create_tensor: loading tensor blk.9.attn_norm.weight
create_tensor: loading tensor blk.9.attn_q.weight
create_tensor: loading tensor blk.9.attn_k.weight
create_tensor: loading tensor blk.9.attn_v.weight
create_tensor: loading tensor blk.9.attn_q.bias
create_tensor: loading tensor blk.9.attn_k.bias
create_tensor: loading tensor blk.9.attn_v.bias
create_tensor: loading tensor blk.9.attn_output.weight
create_tensor: loading tensor blk.9.ffn_norm.weight
create_tensor: loading tensor blk.9.ffn_gate.weight
create_tensor: loading tensor blk.9.ffn_down.weight
create_tensor: loading tensor blk.9.ffn_up.weight
create_tensor: loading tensor blk.10.attn_norm.weight
create_tensor: loading tensor blk.10.attn_q.weight
create_tensor: loading tensor blk.10.attn_k.weight
create_tensor: loading tensor blk.10.attn_v.weight
create_tensor: loading tensor blk.10.attn_q.bias
create_tensor: loading tensor blk.10.attn_k.bias
create_tensor: loading tensor blk.10.attn_v.bias
create_tensor: loading tensor blk.10.attn_output.weight
create_tensor: loading tensor blk.10.ffn_norm.weight
create_tensor: loading tensor blk.10.ffn_gate.weight
create_tensor: loading tensor blk.10.ffn_down.weight
create_tensor: loading tensor blk.10.ffn_up.weight
create_tensor: loading tensor blk.11.attn_norm.weight
create_tensor: loading tensor blk.11.attn_q.weight
create_tensor: loading tensor blk.11.attn_k.weight
create_tensor: loading tensor blk.11.attn_v.weight
create_tensor: loading tensor blk.11.attn_q.bias
create_tensor: loading tensor blk.11.attn_k.bias
create_tensor: loading tensor blk.11.attn_v.bias
create_tensor: loading tensor blk.11.attn_output.weight
create_tensor: loading tensor blk.11.ffn_norm.weight
create_tensor: loading tensor blk.11.ffn_gate.weight
create_tensor: loading tensor blk.11.ffn_down.weight
create_tensor: loading tensor blk.11.ffn_up.weight
create_tensor: loading tensor blk.12.attn_norm.weight
create_tensor: loading tensor blk.12.attn_q.weight
create_tensor: loading tensor blk.12.attn_k.weight
create_tensor: loading tensor blk.12.attn_v.weight
create_tensor: loading tensor blk.12.attn_q.bias
create_tensor: loading tensor blk.12.attn_k.bias
create_tensor: loading tensor blk.12.attn_v.bias
create_tensor: loading tensor blk.12.attn_output.weight
create_tensor: loading tensor blk.12.ffn_norm.weight
create_tensor: loading tensor blk.12.ffn_gate.weight
create_tensor: loading tensor blk.12.ffn_down.weight
create_tensor: loading tensor blk.12.ffn_up.weight
create_tensor: loading tensor blk.13.attn_norm.weight
create_tensor: loading tensor blk.13.attn_q.weight
create_tensor: loading tensor blk.13.attn_k.weight
create_tensor: loading tensor blk.13.attn_v.weight
create_tensor: loading tensor blk.13.attn_q.bias
create_tensor: loading tensor blk.13.attn_k.bias
create_tensor: loading tensor blk.13.attn_v.bias
create_tensor: loading tensor blk.13.attn_output.weight
create_tensor: loading tensor blk.13.ffn_norm.weight
create_tensor: loading tensor blk.13.ffn_gate.weight
create_tensor: loading tensor blk.13.ffn_down.weight
create_tensor: loading tensor blk.13.ffn_up.weight
create_tensor: loading tensor blk.14.attn_norm.weight
create_tensor: loading tensor blk.14.attn_q.weight
create_tensor: loading tensor blk.14.attn_k.weight
create_tensor: loading tensor blk.14.attn_v.weight
create_tensor: loading tensor blk.14.attn_q.bias
create_tensor: loading tensor blk.14.attn_k.bias
create_tensor: loading tensor blk.14.attn_v.bias
create_tensor: loading tensor blk.14.attn_output.weight
create_tensor: loading tensor blk.14.ffn_norm.weight
create_tensor: loading tensor blk.14.ffn_gate.weight
create_tensor: loading tensor blk.14.ffn_down.weight
create_tensor: loading tensor blk.14.ffn_up.weight
create_tensor: loading tensor blk.15.attn_norm.weight
create_tensor: loading tensor blk.15.attn_q.weight
create_tensor: loading tensor blk.15.attn_k.weight
create_tensor: loading tensor blk.15.attn_v.weight
create_tensor: loading tensor blk.15.attn_q.bias
create_tensor: loading tensor blk.15.attn_k.bias
create_tensor: loading tensor blk.15.attn_v.bias
create_tensor: loading tensor blk.15.attn_output.weight
create_tensor: loading tensor blk.15.ffn_norm.weight
create_tensor: loading tensor blk.15.ffn_gate.weight
create_tensor: loading tensor blk.15.ffn_down.weight
create_tensor: loading tensor blk.15.ffn_up.weight
create_tensor: loading tensor blk.16.attn_norm.weight
create_tensor: loading tensor blk.16.attn_q.weight
create_tensor: loading tensor blk.16.attn_k.weight
create_tensor: loading tensor blk.16.attn_v.weight
create_tensor: loading tensor blk.16.attn_q.bias
create_tensor: loading tensor blk.16.attn_k.bias
create_tensor: loading tensor blk.16.attn_v.bias
create_tensor: loading tensor blk.16.attn_output.weight
create_tensor: loading tensor blk.16.ffn_norm.weight
create_tensor: loading tensor blk.16.ffn_gate.weight
create_tensor: loading tensor blk.16.ffn_down.weight
create_tensor: loading tensor blk.16.ffn_up.weight
create_tensor: loading tensor blk.17.attn_norm.weight
create_tensor: loading tensor blk.17.attn_q.weight
create_tensor: loading tensor blk.17.attn_k.weight
create_tensor: loading tensor blk.17.attn_v.weight
create_tensor: loading tensor blk.17.attn_q.bias
create_tensor: loading tensor blk.17.attn_k.bias
create_tensor: loading tensor blk.17.attn_v.bias
create_tensor: loading tensor blk.17.attn_output.weight
create_tensor: loading tensor blk.17.ffn_norm.weight
create_tensor: loading tensor blk.17.ffn_gate.weight
create_tensor: loading tensor blk.17.ffn_down.weight
create_tensor: loading tensor blk.17.ffn_up.weight
create_tensor: loading tensor blk.18.attn_norm.weight
create_tensor: loading tensor blk.18.attn_q.weight
create_tensor: loading tensor blk.18.attn_k.weight
create_tensor: loading tensor blk.18.attn_v.weight
create_tensor: loading tensor blk.18.attn_q.bias
create_tensor: loading tensor blk.18.attn_k.bias
create_tensor: loading tensor blk.18.attn_v.bias
create_tensor: loading tensor blk.18.attn_output.weight
create_tensor: loading tensor blk.18.ffn_norm.weight
create_tensor: loading tensor blk.18.ffn_gate.weight
create_tensor: loading tensor blk.18.ffn_down.weight
create_tensor: loading tensor blk.18.ffn_up.weight
create_tensor: loading tensor blk.19.attn_norm.weight
create_tensor: loading tensor blk.19.attn_q.weight
create_tensor: loading tensor blk.19.attn_k.weight
create_tensor: loading tensor blk.19.attn_v.weight
create_tensor: loading tensor blk.19.attn_q.bias
create_tensor: loading tensor blk.19.attn_k.bias
create_tensor: loading tensor blk.19.attn_v.bias
create_tensor: loading tensor blk.19.attn_output.weight
create_tensor: loading tensor blk.19.ffn_norm.weight
create_tensor: loading tensor blk.19.ffn_gate.weight
create_tensor: loading tensor blk.19.ffn_down.weight
create_tensor: loading tensor blk.19.ffn_up.weight
create_tensor: loading tensor blk.20.attn_norm.weight
create_tensor: loading tensor blk.20.attn_q.weight
create_tensor: loading tensor blk.20.attn_k.weight
create_tensor: loading tensor blk.20.attn_v.weight
create_tensor: loading tensor blk.20.attn_q.bias
create_tensor: loading tensor blk.20.attn_k.bias
create_tensor: loading tensor blk.20.attn_v.bias
create_tensor: loading tensor blk.20.attn_output.weight
create_tensor: loading tensor blk.20.ffn_norm.weight
create_tensor: loading tensor blk.20.ffn_gate.weight
create_tensor: loading tensor blk.20.ffn_down.weight
create_tensor: loading tensor blk.20.ffn_up.weight
create_tensor: loading tensor blk.21.attn_norm.weight
create_tensor: loading tensor blk.21.attn_q.weight
create_tensor: loading tensor blk.21.attn_k.weight
create_tensor: loading tensor blk.21.attn_v.weight
create_tensor: loading tensor blk.21.attn_q.bias
create_tensor: loading tensor blk.21.attn_k.bias
create_tensor: loading tensor blk.21.attn_v.bias
create_tensor: loading tensor blk.21.attn_output.weight
create_tensor: loading tensor blk.21.ffn_norm.weight
create_tensor: loading tensor blk.21.ffn_gate.weight
create_tensor: loading tensor blk.21.ffn_down.weight
create_tensor: loading tensor blk.21.ffn_up.weight
create_tensor: loading tensor blk.22.attn_norm.weight
create_tensor: loading tensor blk.22.attn_q.weight
create_tensor: loading tensor blk.22.attn_k.weight
create_tensor: loading tensor blk.22.attn_v.weight
create_tensor: loading tensor blk.22.attn_q.bias
create_tensor: loading tensor blk.22.attn_k.bias
create_tensor: loading tensor blk.22.attn_v.bias
create_tensor: loading tensor blk.22.attn_output.weight
create_tensor: loading tensor blk.22.ffn_norm.weight
create_tensor: loading tensor blk.22.ffn_gate.weight
create_tensor: loading tensor blk.22.ffn_down.weight
create_tensor: loading tensor blk.22.ffn_up.weight
create_tensor: loading tensor blk.23.attn_norm.weight
create_tensor: loading tensor blk.23.attn_q.weight
create_tensor: loading tensor blk.23.attn_k.weight
create_tensor: loading tensor blk.23.attn_v.weight
create_tensor: loading tensor blk.23.attn_q.bias
create_tensor: loading tensor blk.23.attn_k.bias
create_tensor: loading tensor blk.23.attn_v.bias
create_tensor: loading tensor blk.23.attn_output.weight
create_tensor: loading tensor blk.23.ffn_norm.weight
create_tensor: loading tensor blk.23.ffn_gate.weight
create_tensor: loading tensor blk.23.ffn_down.weight
create_tensor: loading tensor blk.23.ffn_up.weight
create_tensor: loading tensor blk.24.attn_norm.weight
create_tensor: loading tensor blk.24.attn_q.weight
create_tensor: loading tensor blk.24.attn_k.weight
create_tensor: loading tensor blk.24.attn_v.weight
create_tensor: loading tensor blk.24.attn_q.bias
create_tensor: loading tensor blk.24.attn_k.bias
create_tensor: loading tensor blk.24.attn_v.bias
create_tensor: loading tensor blk.24.attn_output.weight
create_tensor: loading tensor blk.24.ffn_norm.weight
create_tensor: loading tensor blk.24.ffn_gate.weight
create_tensor: loading tensor blk.24.ffn_down.weight
create_tensor: loading tensor blk.24.ffn_up.weight
create_tensor: loading tensor blk.25.attn_norm.weight
create_tensor: loading tensor blk.25.attn_q.weight
create_tensor: loading tensor blk.25.attn_k.weight
create_tensor: loading tensor blk.25.attn_v.weight
create_tensor: loading tensor blk.25.attn_q.bias
create_tensor: loading tensor blk.25.attn_k.bias
create_tensor: loading tensor blk.25.attn_v.bias
create_tensor: loading tensor blk.25.attn_output.weight
create_tensor: loading tensor blk.25.ffn_norm.weight
create_tensor: loading tensor blk.25.ffn_gate.weight
create_tensor: loading tensor blk.25.ffn_down.weight
create_tensor: loading tensor blk.25.ffn_up.weight
create_tensor: loading tensor blk.26.attn_norm.weight
create_tensor: loading tensor blk.26.attn_q.weight
create_tensor: loading tensor blk.26.attn_k.weight
create_tensor: loading tensor blk.26.attn_v.weight
create_tensor: loading tensor blk.26.attn_q.bias
create_tensor: loading tensor blk.26.attn_k.bias
create_tensor: loading tensor blk.26.attn_v.bias
create_tensor: loading tensor blk.26.attn_output.weight
create_tensor: loading tensor blk.26.ffn_norm.weight
create_tensor: loading tensor blk.26.ffn_gate.weight
create_tensor: loading tensor blk.26.ffn_down.weight
create_tensor: loading tensor blk.26.ffn_up.weight
create_tensor: loading tensor blk.27.attn_norm.weight
create_tensor: loading tensor blk.27.attn_q.weight
create_tensor: loading tensor blk.27.attn_k.weight
create_tensor: loading tensor blk.27.attn_v.weight
create_tensor: loading tensor blk.27.attn_q.bias
create_tensor: loading tensor blk.27.attn_k.bias
create_tensor: loading tensor blk.27.attn_v.bias
create_tensor: loading tensor blk.27.attn_output.weight
create_tensor: loading tensor blk.27.ffn_norm.weight
create_tensor: loading tensor blk.27.ffn_gate.weight
create_tensor: loading tensor blk.27.ffn_down.weight
create_tensor: loading tensor blk.27.ffn_up.weight
done_getting_tensors: tensor 'token_embd.weight' (q6_K) (and 170 others) cannot be used with preferred buffer type CPU_REPACK, using CPU instead
load_tensors:   CPU_Mapped model buffer size =   934.69 MiB
load_tensors:   CPU_REPACK model buffer size =   596.53 MiB
................repack: repack tensor blk.0.attn_q.weight with q4_K_8x8
repack: repack tensor blk.0.attn_k.weight with q4_K_8x8
repack: repack tensor blk.0.attn_output.weight with q4_K_8x8
repack: repack tensor blk.0.ffn_gate.weight with q4_K_8x8
.repack: repack tensor blk.0.ffn_up.weight with q4_K_8x8
.repack: repack tensor blk.1.attn_q.weight with q4_K_8x8
repack: repack tensor blk.1.attn_k.weight with q4_K_8x8
repack: repack tensor blk.1.attn_output.weight with q4_K_8x8
repack: repack tensor blk.1.ffn_gate.weight with q4_K_8x8
.repack: repack tensor blk.1.ffn_up.weight with q4_K_8x8
repack: repack tensor blk.2.attn_q.weight with q4_K_8x8
.repack: repack tensor blk.2.attn_k.weight with q4_K_8x8
repack: repack tensor blk.2.attn_v.weight with q4_K_8x8
repack: repack tensor blk.2.attn_output.weight with q4_K_8x8
repack: repack tensor blk.2.ffn_gate.weight with q4_K_8x8
.repack: repack tensor blk.2.ffn_down.weight with q4_K_8x8
repack: repack tensor blk.2.ffn_up.weight with q4_K_8x8
.repack: repack tensor blk.3.attn_q.weight with q4_K_8x8
repack: repack tensor blk.3.attn_k.weight with q4_K_8x8
repack: repack tensor blk.3.attn_v.weight with q4_K_8x8
repack: repack tensor blk.3.attn_output.weight with q4_K_8x8
repack: repack tensor blk.3.ffn_gate.weight with q4_K_8x8
.repack: repack tensor blk.3.ffn_down.weight with q4_K_8x8
.repack: repack tensor blk.3.ffn_up.weight with q4_K_8x8
.repack: repack tensor blk.4.attn_q.weight with q4_K_8x8
repack: repack tensor blk.4.attn_k.weight with q4_K_8x8
repack: repack tensor blk.4.attn_v.weight with q4_K_8x8
repack: repack tensor blk.4.attn_output.weight with q4_K_8x8
repack: repack tensor blk.4.ffn_gate.weight with q4_K_8x8
.repack: repack tensor blk.4.ffn_down.weight with q4_K_8x8
.repack: repack tensor blk.4.ffn_up.weight with q4_K_8x8
repack: repack tensor blk.5.attn_q.weight with q4_K_8x8
.repack: repack tensor blk.5.attn_k.weight with q4_K_8x8
repack: repack tensor blk.5.attn_output.weight with q4_K_8x8
repack: repack tensor blk.5.ffn_gate.weight with q4_K_8x8
.repack: repack tensor blk.5.ffn_up.weight with q4_K_8x8
repack: repack tensor blk.6.attn_q.weight with q4_K_8x8
repack: repack tensor blk.6.attn_k.weight with q4_K_8x8
.repack: repack tensor blk.6.attn_output.weight with q4_K_8x8
repack: repack tensor blk.6.ffn_gate.weight with q4_K_8x8
repack: repack tensor blk.6.ffn_up.weight with q4_K_8x8
.repack: repack tensor blk.7.attn_q.weight with q4_K_8x8
repack: repack tensor blk.7.attn_k.weight with q4_K_8x8
repack: repack tensor blk.7.attn_output.weight with q4_K_8x8
.repack: repack tensor blk.7.ffn_gate.weight with q4_K_8x8
repack: repack tensor blk.7.ffn_up.weight with q4_K_8x8
.repack: repack tensor blk.8.attn_q.weight with q4_K_8x8
repack: repack tensor blk.8.attn_k.weight with q4_K_8x8
repack: repack tensor blk.8.attn_output.weight with q4_K_8x8
repack: repack tensor blk.8.ffn_gate.weight with q4_K_8x8
.repack: repack tensor blk.8.ffn_up.weight with q4_K_8x8
.repack: repack tensor blk.9.attn_q.weight with q4_K_8x8
repack: repack tensor blk.9.attn_k.weight with q4_K_8x8
repack: repack tensor blk.9.attn_output.weight with q4_K_8x8
repack: repack tensor blk.9.ffn_gate.weight with q4_K_8x8
.repack: repack tensor blk.9.ffn_up.weight with q4_K_8x8
.repack: repack tensor blk.10.attn_q.weight with q4_K_8x8
repack: repack tensor blk.10.attn_k.weight with q4_K_8x8
repack: repack tensor blk.10.attn_output.weight with q4_K_8x8
repack: repack tensor blk.10.ffn_gate.weight with q4_K_8x8
.repack: repack tensor blk.10.ffn_up.weight with q4_K_8x8
.repack: repack tensor blk.11.attn_q.weight with q4_K_8x8
repack: repack tensor blk.11.attn_k.weight with q4_K_8x8
repack: repack tensor blk.11.attn_v.weight with q4_K_8x8
repack: repack tensor blk.11.attn_output.weight with q4_K_8x8
repack: repack tensor blk.11.ffn_gate.weight with q4_K_8x8
.repack: repack tensor blk.11.ffn_down.weight with q4_K_8x8
.repack: repack tensor blk.11.ffn_up.weight with q4_K_8x8
repack: repack tensor blk.12.attn_q.weight with q4_K_8x8
.repack: repack tensor blk.12.attn_k.weight with q4_K_8x8
repack: repack tensor blk.12.attn_v.weight with q4_K_8x8
repack: repack tensor blk.12.attn_output.weight with q4_K_8x8
repack: repack tensor blk.12.ffn_gate.weight with q4_K_8x8
.repack: repack tensor blk.12.ffn_down.weight with q4_K_8x8
repack: repack tensor blk.12.ffn_up.weight with q4_K_8x8
.repack: repack tensor blk.13.attn_q.weight with q4_K_8x8
repack: repack tensor blk.13.attn_k.weight with q4_K_8x8
repack: repack tensor blk.13.attn_output.weight with q4_K_8x8
repack: repack tensor blk.13.ffn_gate.weight with q4_K_8x8
.repack: repack tensor blk.13.ffn_up.weight with q4_K_8x8
.repack: repack tensor blk.14.attn_q.weight with q4_K_8x8
repack: repack tensor blk.14.attn_k.weight with q4_K_8x8
repack: repack tensor blk.14.attn_v.weight with q4_K_8x8
repack: repack tensor blk.14.attn_output.weight with q4_K_8x8
repack: repack tensor blk.14.ffn_gate.weight with q4_K_8x8
.repack: repack tensor blk.14.ffn_down.weight with q4_K_8x8
.repack: repack tensor blk.14.ffn_up.weight with q4_K_8x8
.repack: repack tensor blk.15.attn_q.weight with q4_K_8x8
repack: repack tensor blk.15.attn_k.weight with q4_K_8x8
repack: repack tensor blk.15.attn_v.weight with q4_K_8x8
repack: repack tensor blk.15.attn_output.weight with q4_K_8x8
repack: repack tensor blk.15.ffn_gate.weight with q4_K_8x8
.repack: repack tensor blk.15.ffn_down.weight with q4_K_8x8
.repack: repack tensor blk.15.ffn_up.weight with q4_K_8x8
repack: repack tensor blk.16.attn_q.weight with q4_K_8x8
repack: repack tensor blk.16.attn_k.weight with q4_K_8x8
repack: repack tensor blk.16.attn_output.weight with q4_K_8x8
.repack: repack tensor blk.16.ffn_gate.weight with q4_K_8x8
repack: repack tensor blk.16.ffn_up.weight with q4_K_8x8
.repack: repack tensor blk.17.attn_q.weight with q4_K_8x8
repack: repack tensor blk.17.attn_k.weight with q4_K_8x8
repack: repack tensor blk.17.attn_v.weight with q4_K_8x8
repack: repack tensor blk.17.attn_output.weight with q4_K_8x8
.repack: repack tensor blk.17.ffn_gate.weight with q4_K_8x8
repack: repack tensor blk.17.ffn_down.weight with q4_K_8x8
.repack: repack tensor blk.17.ffn_up.weight with q4_K_8x8
.repack: repack tensor blk.18.attn_q.weight with q4_K_8x8
repack: repack tensor blk.18.attn_k.weight with q4_K_8x8
repack: repack tensor blk.18.attn_v.weight with q4_K_8x8
repack: repack tensor blk.18.attn_output.weight with q4_K_8x8
repack: repack tensor blk.18.ffn_gate.weight with q4_K_8x8
.repack: repack tensor blk.18.ffn_down.weight with q4_K_8x8
.repack: repack tensor blk.18.ffn_up.weight with q4_K_8x8
.repack: repack tensor blk.19.attn_q.weight with q4_K_8x8
repack: repack tensor blk.19.attn_k.weight with q4_K_8x8
repack: repack tensor blk.19.attn_output.weight with q4_K_8x8
repack: repack tensor blk.19.ffn_gate.weight with q4_K_8x8
.repack: repack tensor blk.19.ffn_up.weight with q4_K_8x8
repack: repack tensor blk.20.attn_q.weight with q4_K_8x8
.repack: repack tensor blk.20.attn_k.weight with q4_K_8x8
repack: repack tensor blk.20.attn_v.weight with q4_K_8x8
repack: repack tensor blk.20.attn_output.weight with q4_K_8x8
repack: repack tensor blk.20.ffn_gate.weight with q4_K_8x8
.repack: repack tensor blk.20.ffn_down.weight with q4_K_8x8
repack: repack tensor blk.20.ffn_up.weight with q4_K_8x8
.repack: repack tensor blk.21.attn_q.weight with q4_K_8x8
repack: repack tensor blk.21.attn_k.weight with q4_K_8x8
repack: repack tensor blk.21.attn_output.weight with q4_K_8x8
repack: repack tensor blk.21.ffn_gate.weight with q4_K_8x8
.repack: repack tensor blk.21.ffn_up.weight with q4_K_8x8
.repack: repack tensor blk.22.attn_q.weight with q4_K_8x8
repack: repack tensor blk.22.attn_k.weight with q4_K_8x8
repack: repack tensor blk.22.attn_v.weight with q4_K_8x8
repack: repack tensor blk.22.attn_output.weight with q4_K_8x8
repack: repack tensor blk.22.ffn_gate.weight with q4_K_8x8
.repack: repack tensor blk.22.ffn_down.weight with q4_K_8x8
.repack: repack tensor blk.22.ffn_up.weight with q4_K_8x8
.repack: repack tensor blk.23.attn_q.weight with q4_K_8x8
repack: repack tensor blk.23.attn_k.weight with q4_K_8x8
repack: repack tensor blk.23.attn_v.weight with q4_K_8x8
repack: repack tensor blk.23.attn_output.weight with q4_K_8x8
repack: repack tensor blk.23.ffn_gate.weight with q4_K_8x8
.repack: repack tensor blk.23.ffn_down.weight with q4_K_8x8
.repack: repack tensor blk.23.ffn_up.weight with q4_K_8x8
repack: repack tensor blk.24.attn_q.weight with q4_K_8x8
.repack: repack tensor blk.24.attn_k.weight with q4_K_8x8
repack: repack tensor blk.24.attn_output.weight with q4_K_8x8
repack: repack tensor blk.24.ffn_gate.weight with q4_K_8x8
repack: repack tensor blk.24.ffn_up.weight with q4_K_8x8
.repack: repack tensor blk.25.attn_q.weight with q4_K_8x8
repack: repack tensor blk.25.attn_k.weight with q4_K_8x8
repack: repack tensor blk.25.attn_v.weight with q4_K_8x8
repack: repack tensor blk.25.attn_output.weight with q4_K_8x8
.repack: repack tensor blk.25.ffn_gate.weight with q4_K_8x8
repack: repack tensor blk.25.ffn_down.weight with q4_K_8x8
.repack: repack tensor blk.25.ffn_up.weight with q4_K_8x8
.repack: repack tensor blk.26.attn_q.weight with q4_K_8x8
repack: repack tensor blk.26.attn_k.weight with q4_K_8x8
repack: repack tensor blk.26.attn_v.weight with q4_K_8x8
repack: repack tensor blk.26.attn_output.weight with q4_K_8x8
repack: repack tensor blk.26.ffn_gate.weight with q4_K_8x8
.repack: repack tensor blk.26.ffn_down.weight with q4_K_8x8
.repack: repack tensor blk.26.ffn_up.weight with q4_K_8x8
.repack: repack tensor blk.27.attn_q.weight with q4_K_8x8
repack: repack tensor blk.27.attn_k.weight with q4_K_8x8
repack: repack tensor blk.27.attn_output.weight with q4_K_8x8
repack: repack tensor blk.27.ffn_gate.weight with q4_K_8x8
.repack: repack tensor blk.27.ffn_up.weight with q4_K_8x8
.
llama_context: constructing llama_context
llama_context: n_seq_max     = 1
llama_context: n_ctx         = 8192
llama_context: n_ctx_seq     = 8192
llama_context: n_batch       = 512
llama_context: n_ubatch      = 512
llama_context: causal_attn   = 1
llama_context: flash_attn    = disabled
llama_context: kv_unified    = false
llama_context: freq_base     = 1000000.0
llama_context: freq_scale    = 1
llama_context: n_rs_seq      = 0
llama_context: n_outputs_max = 512
llama_context: n_ctx_seq (8192) < n_ctx_train (32768) -- the full capacity of the model will not be utilized
set_abort_callback: call
llama_context:        CPU  output buffer size =     0.58 MiB
llama_kv_cache: layer   0: dev = CPU
llama_kv_cache: layer   1: dev = CPU
llama_kv_cache: layer   2: dev = CPU
llama_kv_cache: layer   3: dev = CPU
llama_kv_cache: layer   4: dev = CPU
llama_kv_cache: layer   5: dev = CPU
llama_kv_cache: layer   6: dev = CPU
llama_kv_cache: layer   7: dev = CPU
llama_kv_cache: layer   8: dev = CPU
llama_kv_cache: layer   9: dev = CPU
llama_kv_cache: layer  10: dev = CPU
llama_kv_cache: layer  11: dev = CPU
llama_kv_cache: layer  12: dev = CPU
llama_kv_cache: layer  13: dev = CPU
llama_kv_cache: layer  14: dev = CPU
llama_kv_cache: layer  15: dev = CPU
llama_kv_cache: layer  16: dev = CPU
llama_kv_cache: layer  17: dev = CPU
llama_kv_cache: layer  18: dev = CPU
llama_kv_cache: layer  19: dev = CPU
llama_kv_cache: layer  20: dev = CPU
llama_kv_cache: layer  21: dev = CPU
llama_kv_cache: layer  22: dev = CPU
llama_kv_cache: layer  23: dev = CPU
llama_kv_cache: layer  24: dev = CPU
llama_kv_cache: layer  25: dev = CPU
llama_kv_cache: layer  26: dev = CPU
llama_kv_cache: layer  27: dev = CPU
llama_kv_cache:        CPU KV buffer size =   224.00 MiB
llama_kv_cache: size =  224.00 MiB (  8192 cells,  28 layers,  1/1 seqs), K (f16):  112.00 MiB, V (f16):  112.00 MiB
llama_kv_cache: attn_rot_k = 0, n_embd_head_k_all = 128
llama_kv_cache: attn_rot_v = 0, n_embd_head_k_all = 128
llama_context: enumerating backends
llama_context: backend_ptrs.size() = 1
sched_reserve: reserving ...
sched_reserve: max_nodes = 2704
sched_reserve: reserving full memory module
sched_reserve: worst-case: n_tokens = 512, n_seqs = 1, n_outputs = 1
resolve_fused_ops: resolving fused Gated Delta Net support:
graph_reserve: reserving a graph for ubatch with n_tokens =    1, n_seqs =  1, n_outputs =    1
resolve_fused_ops: fused Gated Delta Net (autoregressive) enabled
graph_reserve: reserving a graph for ubatch with n_tokens =   16, n_seqs =  1, n_outputs =   16
resolve_fused_ops: fused Gated Delta Net (chunked) enabled
resolve_fused_ops: resolving fused Lightning Indexer support:
graph_reserve: reserving a graph for ubatch with n_tokens =    1, n_seqs =  1, n_outputs =    1
resolve_fused_ops: Lightning Indexer enabled
graph_reserve: reserving a graph for ubatch with n_tokens =  512, n_seqs =  1, n_outputs =  512
graph_reserve: reserving a graph for ubatch with n_tokens =    1, n_seqs =  1, n_outputs =    1
graph_reserve: reserving a graph for ubatch with n_tokens =  512, n_seqs =  1, n_outputs =  512
sched_reserve:        CPU compute buffer size =   305.75 MiB
sched_reserve: graph nodes  = 1098
sched_reserve: graph splits = 1
sched_reserve: reserve took 50.16 ms, sched copies = 1
CPU : SSE3 = 1 | SSSE3 = 1 | AVX = 1 | AVX2 = 1 | F16C = 1 | FMA = 1 | BMI2 = 1 | LLAMAFILE = 1 | OPENMP = 1 | REPACK = 1 |
Model metadata: {'quantize.imatrix.chunks_count': '128', 'quantize.imatrix.entries_count': '196', 'quantize.imatrix.file': '/models_out/Qwen2.5-1.5B-Instruct-GGUF/Qwen2.5-1.5B-Instruct.imatrix', 'general.base_model.0.repo_url': 'https://huggingface.co/Qwen/Qwen2.5-1.5B', 'general.license': 'apache-2.0', 'qwen2.attention.head_count_kv': '2', 'tokenizer.chat_template': '{%- if tools %}\n    {{- \'<|im_start|>system\\n\' }}\n    {%- if messages[0][\'role\'] == \'system\' %}\n        {{- messages[0][\'content\'] }}\n    {%- else %}\n        {{- \'You are Qwen, created by Alibaba Cloud. You are a helpful assistant.\' }}\n    {%- endif %}\n    {{- "\\n\\n# Tools\\n\\nYou may call one or more functions to assist with the user query.\\n\\nYou are provided with function signatures within <tools></tools> XML tags:\\n<tools>" }}\n    {%- for tool in tools %}\n        {{- "\\n" }}\n        {{- tool | tojson }}\n    {%- endfor %}\n    {{- "\\n</tools>\\n\\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\\n<tool_call>\\n{\\"name\\": <function-name>, \\"arguments\\": <args-json-object>}\\n</tool_call><|im_end|>\\n" }}\n{%- else %}\n    {%- if messages[0][\'role\'] == \'system\' %}\n        {{- \'<|im_start|>system\\n\' + messages[0][\'content\'] + \'<|im_end|>\\n\' }}\n    {%- else %}\n        {{- \'<|im_start|>system\\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\\n\' }}\n    {%- endif %}\n{%- endif %}\n{%- for message in messages %}\n    {%- if (message.role == "user") or (message.role == "system" and not loop.first) or (message.role == "assistant" and not message.tool_calls) %}\n        {{- \'<|im_start|>\' + message.role + \'\\n\' + message.content + \'<|im_end|>\' + \'\\n\' }}\n    {%- elif message.role == "assistant" %}\n        {{- \'<|im_start|>\' + message.role }}\n        {%- if message.content %}\n            {{- \'\\n\' + message.content }}\n        {%- endif %}\n        {%- for tool_call in message.tool_calls %}\n            {%- if tool_call.function is defined %}\n                {%- set tool_call = tool_call.function %}\n            {%- endif %}\n            {{- \'\\n<tool_call>\\n{"name": "\' }}\n            {{- tool_call.name }}\n            {{- \'", "arguments": \' }}\n            {{- tool_call.arguments | tojson }}\n            {{- \'}\\n</tool_call>\' }}\n        {%- endfor %}\n        {{- \'<|im_end|>\\n\' }}\n    {%- elif message.role == "tool" %}\n        {%- if (loop.index0 == 0) or (messages[loop.index0 - 1].role != "tool") %}\n            {{- \'<|im_start|>user\' }}\n        {%- endif %}\n        {{- \'\\n<tool_response>\\n\' }}\n        {{- message.content }}\n        {{- \'\\n</tool_response>\' }}\n        {%- if loop.last or (messages[loop.index0 + 1].role != "tool") %}\n            {{- \'<|im_end|>\\n\' }}\n        {%- endif %}\n    {%- endif %}\n{%- endfor %}\n{%- if add_generation_prompt %}\n    {{- \'<|im_start|>assistant\\n\' }}\n{%- endif %}\n', 'general.type': 'model', 'qwen2.block_count': '28', 'quantize.imatrix.dataset': '/training_dir/calibration_datav3.txt', 'general.base_model.0.name': 'Qwen2.5 1.5B', 'general.license.link': 'https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct/blob/main/LICENSE', 'tokenizer.ggml.pre': 'qwen2', 'general.base_model.count': '1', 'general.base_model.0.organization': 'Qwen', 'general.size_label': '1.5B', 'tokenizer.ggml.add_bos_token': 'false', 'general.basename': 'Qwen2.5', 'qwen2.embedding_length': '1536', 'tokenizer.ggml.padding_token_id': '151643', 'general.architecture': 'qwen2', 'qwen2.context_length': '32768', 'qwen2.feed_forward_length': '8960', 'tokenizer.ggml.model': 'gpt2', 'general.quantization_version': '2', 'qwen2.attention.head_count': '12', 'qwen2.rope.freq_base': '1000000.000000', 'tokenizer.ggml.eos_token_id': '151645', 'qwen2.attention.layer_norm_rms_epsilon': '0.000001', 'general.finetune': 'Instruct', 'general.file_type': '15', 'general.name': 'Qwen2.5 1.5B Instruct', 'tokenizer.ggml.bos_token_id': '151643'}
Available chat formats from metadata: chat_template.default
Using gguf chat template: {%- if tools %}
    {{- '<|im_start|>system\n' }}
    {%- if messages[0]['role'] == 'system' %}
        {{- messages[0]['content'] }}
    {%- else %}
        {{- 'You are Qwen, created by Alibaba Cloud. You are a helpful assistant.' }}
    {%- endif %}
    {{- "\n\n# Tools\n\nYou may call one or more functions to assist with the user query.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n<tools>" }}
    {%- for tool in tools %}
        {{- "\n" }}
        {{- tool | tojson }}
    {%- endfor %}
    {{- "\n</tools>\n\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\n<tool_call>\n{\"name\": <function-name>, \"arguments\": <args-json-object>}\n</tool_call><|im_end|>\n" }}
{%- else %}
    {%- if messages[0]['role'] == 'system' %}
        {{- '<|im_start|>system\n' + messages[0]['content'] + '<|im_end|>\n' }}
    {%- else %}
        {{- '<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n' }}
    {%- endif %}
{%- endif %}
{%- for message in messages %}
    {%- if (message.role == "user") or (message.role == "system" and not loop.first) or (message.role == "assistant" and not message.tool_calls) %}
        {{- '<|im_start|>' + message.role + '\n' + message.content + '<|im_end|>' + '\n' }}
    {%- elif message.role == "assistant" %}
        {{- '<|im_start|>' + message.role }}
        {%- if message.content %}
            {{- '\n' + message.content }}
        {%- endif %}
        {%- for tool_call in message.tool_calls %}
            {%- if tool_call.function is defined %}
                {%- set tool_call = tool_call.function %}
            {%- endif %}
            {{- '\n<tool_call>\n{"name": "' }}
            {{- tool_call.name }}
            {{- '", "arguments": ' }}
            {{- tool_call.arguments | tojson }}
            {{- '}\n</tool_call>' }}
        {%- endfor %}
        {{- '<|im_end|>\n' }}
    {%- elif message.role == "tool" %}
        {%- if (loop.index0 == 0) or (messages[loop.index0 - 1].role != "tool") %}
            {{- '<|im_start|>user' }}
        {%- endif %}
        {{- '\n<tool_response>\n' }}
        {{- message.content }}
        {{- '\n</tool_response>' }}
        {%- if loop.last or (messages[loop.index0 + 1].role != "tool") %}
            {{- '<|im_end|>\n' }}
        {%- endif %}
    {%- endif %}
{%- endfor %}
{%- if add_generation_prompt %}
    {{- '<|im_start|>assistant\n' }}
{%- endif %}

Using chat eos_token: <|im_end|>
Using chat bos_token: <|endoftext|>

--- بدأ Qwen في البحث والحساب ---
╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Search for the current Bitcoin price in USD and convert it to Egyptian Pounds (assume 1 USD = 50 EGP).          │
│                                                                                                                 │
╰─ QwenGGUFModel - Qwen2.5-Local ─────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

llama_perf_context_print:        load time =  135843.17 ms
llama_perf_context_print: prompt eval time =  135841.64 ms /  2106 tokens (   64.50 ms per token,    15.50 tokens per second)
llama_perf_context_print:        eval time =   22379.85 ms /   128 runs   (  174.84 ms per token,     5.72 tokens per second)
llama_perf_context_print:       total time =  158365.74 ms /  2234 tokens
llama_perf_context_print:    graphs reused =        127
 ─ Executing parsed code: ────────────────────────────────────────────────────────────────────────────────────────
  bitcoin_price_usd = web_search(query="current Bitcoin price in USD")                                             
  print("Current Bitcoin price in USD:", bitcoin_price_usd)                                                        
  bitcoin_price_egp = currency_converter(amount=bitcoin_price_usd, from_currency="USD", to_currency="EGP")         
  print("Current Bitcoin price in EGP:", bitcoin_price_egp)                                                        
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Current Bitcoin price in USD: ## Search Results

[Bitcoin BTC (BTC-USD) Live Price, News, Chart & Price History - Yahoo
Finance](https://finance.yahoo.com/quote/BTC-USD/)
1 day ago - Users are able to generate BTC through the process of mining. Bitcoin has a current supply of
20,065,587. The last known price of Bitcoin is 63,437.72830162 USD and is up 1.34 over the last 24 hours. It is
currently trading on 12682 active market(s) with $25,552,248,772.52 traded over the last ...

[Bitcoin Price | BTC to USD Converter, Chart and News](https://www.binance.com/en/price/bitcoin)
June 30, 2026 - Live price of Bitcoin is $96,262.14 with a market cap of $1,908.19B USD. Discover current price,
trading volume, chart history, and more.

[Bitcoin price today, BTC to USD live price, marketcap and chart |
CoinMarketCap](https://coinmarketcap.com/currencies/bitcoin/)
2 days ago - The live Bitcoin price today is $64,680.46 USD with a 24-hour trading volume of $20,501,138,716.65
USD. We update our BTC to USD price in real-time.

[Bitcoin Price Chart (BTC/USD) | Bitcoin Value | bitFlyer USA](https://bitflyer.com/en-us/bitcoin-chart)
Log In Create an Account · Home > Bitcoin Price Chart (BTC/USD) 64,719.72*USD · +0.18 % *The reference price is
calculated using the mid-price at the current point in time. The actual execution price may differ. Ethereum ·
ETH/USD · +1.07 % 1,914.86 USD ·

[Convert Bitcoin to USD | Bitcoin Price in US Dollars | Revolut United
Kingdom](https://www.revolut.com/crypto/price/btc/usd/)
Convert 1 Bitcoin (BTC) to US Dollar (USD) with our instant cryptocurrency converter. 1 BTC is currently worth
$64,677.29. Avoid high fees with Revolut.

[BTC USD — Bitcoin Price and Chart — TradingView](https://www.tradingview.com/symbols/BTCUSD/)
1 week ago - The former support is now acting as resistance, increasing the probability of further downside toward
the marked support levels. The overall market structure stays bearish while price trades below the resista ... The
current price of Bitcoin (BTC) is 64,662 USD — it has fallen −0.15% in ...

[Buy Bitcoin - BTC Price Today, Live Charts and News](https://robinhood.com/us/en/crypto/BTC/)
5 days ago - The price of Bitcoin is $0.00. Buy Bitcoin - BTC with $1. Invest in BTC cryptocurrency with Robinhood
in the easiest and fastest way.

[Bitcoin (BTC) Price USD Today, News, Charts, Market Cap | Coinbase](https://www.coinbase.com/price/bitcoin)
10 hours ago - We update our Bitcoin to USD currency in real-time. Get the live price of Bitcoin on Coinbase. ...
The current market cap of Bitcoin is $1.292T.

[BTC.CM=: Bitcoin/USD Coin Metrics - Stock Price, Quote and News - CNBC](https://www.cnbc.com/quotes/BTC.CM=)
2 weeks ago - Get Bitcoin/USD Coin Metrics (BTC.CM=:Exchange) real-time stock quotes, news, price and financial
information from CNBC.

[Bitcoin price today - BTC price chart & live trends](https://www.kraken.com/prices/bitcoin)
Bitcoin price today is $64,601.00. In the last 24 hours Bitcoin's price moved +0.43%. The current BTC to USD
conversion rate is $64,601.00 per BTC.


Code execution failed at line 'bitcoin_price_egp = currency_converter(amount=bitcoin_price_usd,
from_currency="USD", to_currency="EGP")' due to: InterpreterError: Forbidden function evaluation:
'currency_converter' is not among the explicitly allowed tools or defined/imported in the preceding code

[Step 1: Duration 159.15 seconds]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Llama.generate: 2104 prefix-match hit, remaining 1262 prompt tokens to eval
llama_perf_context_print:        load time =  135843.17 ms
llama_perf_context_print: prompt eval time =   95362.66 ms /  1262 tokens (   75.56 ms per token,    13.23 tokens per second)
llama_perf_context_print:        eval time =   13605.58 ms /    74 runs   (  183.86 ms per token,     5.44 tokens per second)
llama_perf_context_print:       total time =  109041.45 ms /  1336 tokens
llama_perf_context_print:    graphs reused =         73
 ─ Executing parsed code: ────────────────────────────────────────────────────────────────────────────────────────
  bitcoin_to_egp = web_search(query="convert Bitcoin to EGP")                                                      
  print("Bitcoin to EGP conversion tool:", bitcoin_to_egp)                                                         
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Bitcoin to EGP conversion tool: ## Search Results

[BTC to EGP | Convert Bitcoin to Egyptian Pound | OKX](https://www.okx.com/convert/btc-to-egp)
To find out the most recent value of 1 Bitcoin in EGP, consult the conversion tables on this page. This will help
you better understand how to convert Bitcoin into Egyptian Pound and track changes in value.

[BTC to EGP Today | 1 BTC to EGP Live Rate & Converter](https://www.binance.com/en-AU/price/bitcoin/EGP)
Convert Bitcoin (BTC) to Egyptian Pound (EGP) instantly using today's live rate of E£ 3.2M per BTC. Use our
conversion calculator to get a real-time quote to convert Bitcoin (BTC) to Egyptian Pound (EGP).

[Convert Bitcoin to EGP | Bitcoin Price in... | Revolut United
Kingdom](https://www.revolut.com/crypto/price/btc/egp/)
BTC to EGP: Convert Bitcoin (BTC) to Egyptian Pounds (EGP). Find the live price data you need to get confident with
crypto, all in one place.

[1 BTC to EGP - Bitcoins to Egyptian Pounds Exchange
Rate](https://www.xe.com/currencyconverter/convert/?Amount=1&From=BTC&To=EGP)
Get the latest 1 Bitcoin to Egyptian Pound rate for FREE with the original Universal Currency Converter. Set rate
alerts for BTC to EGP and learn more about Bitcoins and Egyptian Pounds from XE - the Currency Authority.

[BTC to EGP: Convert Bitcoin (BTC) to Egyptian Pound (EGP)](https://www.coinbase.com/converter/btc/egp)
Easily convert Bitcoin to Egyptian Pound with our cryptocurrency converter on Coinbase United States. 1 BTC is
currently worth EGP 3,302,805.81.

[BTC to EGP: Convert Bitcoin to Egyptian Pound | Live BTC... | MEXC](https://www.mexc.co/price/BTC/EGP)
BTC/EGP: Access live Bitcoin prices in EGP, real-time exchange rates, and fast crypto-to-fiat conversions.50 EGP
can be converted to 0.0{4}1502 BTC, excluding any platform or gas fees. The conversion rate of 1 BTC to EGP has
changed by -3.17% in the last 7 days.

[1 BTC to EGP Exchange Rate - Bitcoin to Egyptian
Pound](https://paytm.com/tools/currency-converter/amount-1-from-btc-to-egp/)
Converting Bitcoin (BTC) to Egyptian Pound (EGP) offers several advantages, depending on your financial needs and
market conditions. Here are some key benefits: 1. Favorable Exchange Rates.

[Bitcoin Calculator: Convert Bitcoin (BTC) to Egyptian Pound (EGP)](https://www.bitget.com/price/bitcoin/EGP)
Bitget converter provides BTC to EGP real-time exchange rates, making it easy to convert Bitcoin (BTC) to Egyptian
Pound (EGP). The conversion result is based on real-time data.

[1 BTC To EGP - Convert Bitcoin (BTC) to EGP (EGP) | Gate](https://www.gate.com/converter/bitcoin-btc/egp)
Our Bitcoin to EGP (EGP) converter is an online tool that calculates the equivalent value of Bitcoin in EGP. This
tool uses the current market rate to convert the digital currency (Bitcoin) into the fiat currency (EGP).

[1 BTC to EGP - Bitcoin to Egyptian Pound Converter - BitKan.com](https://bitkan.com/convert/btc-to-egp)
Conversion Tables. 1 BTC to EGP Converter Stats — Changes in Egyptian Pound price (EGP denominated). Last 24
hours.The relative change between the highs and lows in Bitcoin price EGP in the last 30 days indicates a
volatility of -3.47%.

Out: None

[Step 2: Duration 110.77 seconds]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 3 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Llama.generate: 3364 prefix-match hit, remaining 1014 prompt tokens to eval
llama_perf_context_print:        load time =  135843.17 ms
llama_perf_context_print: prompt eval time =   85452.67 ms /  1014 tokens (   84.27 ms per token,    11.87 tokens per second)
llama_perf_context_print:        eval time =   17516.37 ms /    93 runs   (  188.35 ms per token,     5.31 tokens per second)
llama_perf_context_print:       total time =  103064.28 ms /  1107 tokens
llama_perf_context_print:    graphs reused =         92
 ─ Executing parsed code: ────────────────────────────────────────────────────────────────────────────────────────
  bitcoin_price_usd = 64677.29                                                                                     
  bitcoin_to_egp = currency_converter(amount=bitcoin_price_usd, from_currency="USD", to_currency="EGP")            
  final_answer(bitcoin_to_egp)                                                                                     
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Code execution failed at line 'bitcoin_to_egp = currency_converter(amount=bitcoin_price_usd, from_currency="USD",
to_currency="EGP")' due to: InterpreterError: Forbidden function evaluation: 'currency_converter' is not among the
explicitly allowed tools or defined/imported in the preceding code

[Step 3: Duration 103.11 seconds]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 4 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Llama.generate: 4376 prefix-match hit, remaining 307 prompt tokens to eval
llama_perf_context_print:        load time =  135843.17 ms
llama_perf_context_print: prompt eval time =   27278.88 ms /   307 tokens (   88.86 ms per token,    11.25 tokens per second)
llama_perf_context_print:        eval time =   16589.95 ms /    74 runs   (  224.19 ms per token,     4.46 tokens per second)
llama_perf_context_print:       total time =   43946.71 ms /   381 tokens
llama_perf_context_print:    graphs reused =         73
 ─ Executing parsed code: ────────────────────────────────────────────────────────────────────────────────────────
  bitcoin_to_egp = web_search(query="convert Bitcoin to EGP")                                                      
  print("Bitcoin to EGP conversion tool:", bitcoin_to_egp)                                                         
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Bitcoin to EGP conversion tool: ## Search Results

[BTC to EGP | Convert Bitcoin to Egyptian Pound | OKX](https://www.okx.com/convert/btc-to-egp)
To find out the most recent value of 1 Bitcoin in EGP, consult the conversion tables on this page. This will help
you better understand how to convert Bitcoin into Egyptian Pound and track changes in value.

[BTC to EGP Today | 1 BTC to EGP Live Rate & Converter](https://www.binance.com/en-AU/price/bitcoin/EGP)
Convert Bitcoin (BTC) to Egyptian Pound (EGP) instantly using today's live rate of E£ 3.2M per BTC. Use our
conversion calculator to get a real-time quote to convert Bitcoin (BTC) to Egyptian Pound (EGP).

[Convert Bitcoin to EGP | Bitcoin Price in... | Revolut United
Kingdom](https://www.revolut.com/crypto/price/btc/egp/)
BTC to EGP: Convert Bitcoin (BTC) to Egyptian Pounds (EGP). Find the live price data you need to get confident with
crypto, all in one place.

[1 BTC to EGP - Bitcoins to Egyptian Pounds Exchange
Rate](https://www.xe.com/currencyconverter/convert/?Amount=1&From=BTC&To=EGP)
Get the latest 1 Bitcoin to Egyptian Pound rate for FREE with the original Universal Currency Converter. Set rate
alerts for BTC to EGP and learn more about Bitcoins and Egyptian Pounds from XE - the Currency Authority.

[BTC to EGP: Convert Bitcoin (BTC) to Egyptian Pound (EGP)](https://www.coinbase.com/converter/btc/egp)
Easily convert Bitcoin to Egyptian Pound with our cryptocurrency converter on Coinbase United States. 1 BTC is
currently worth EGP 3,302,805.81.

[1 BTC to EGP Exchange Rate - Bitcoin to Egyptian
Pound](https://paytm.com/tools/currency-converter/amount-1-from-btc-to-egp/)
Converting Bitcoin (BTC) to Egyptian Pound (EGP) offers several advantages, depending on your financial needs and
market conditions. Here are some key benefits: 1. Favorable Exchange Rates.

[BTC to EGP: Convert Bitcoin to Egyptian Pound | Live BTC... | MEXC](https://www.mexc.co/price/BTC/EGP)
BTC/EGP: Access live Bitcoin prices in EGP, real-time exchange rates, and fast crypto-to-fiat conversions.50 EGP
can be converted to 0.0{4}1502 BTC, excluding any platform or gas fees. The conversion rate of 1 BTC to EGP has
changed by -3.17% in the last 7 days.

[Bitcoin Calculator: Convert Bitcoin (BTC) to Egyptian Pound (EGP)](https://www.bitget.com/price/bitcoin/EGP)
Bitget converter provides BTC to EGP real-time exchange rates, making it easy to convert Bitcoin (BTC) to Egyptian
Pound (EGP). The conversion result is based on real-time data.

[1 BTC To EGP - Convert Bitcoin (BTC) to EGP (EGP) | Gate](https://www.gate.com/converter/bitcoin-btc/egp)
Our Bitcoin to EGP (EGP) converter is an online tool that calculates the equivalent value of Bitcoin in EGP. This
tool uses the current market rate to convert the digital currency (Bitcoin) into the fiat currency (EGP).

[1 BTC to EGP - Bitcoin to Egyptian Pound Converter - BitKan.com](https://bitkan.com/convert/btc-to-egp)
Conversion Tables. 1 BTC to EGP Converter Stats — Changes in Egyptian Pound price (EGP denominated). Last 24
hours.The relative change between the highs and lows in Bitcoin price EGP in the last 30 days indicates a
volatility of -3.47%.

Out: None

[Step 4: Duration 44.89 seconds]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 5 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Llama.generate: 4681 prefix-match hit, remaining 1014 prompt tokens to eval
llama_perf_context_print:        load time =  135843.17 ms
llama_perf_context_print: prompt eval time =   95802.70 ms /  1014 tokens (   94.48 ms per token,    10.58 tokens per second)
llama_perf_context_print:        eval time =   22524.00 ms /    93 runs   (  242.19 ms per token,     4.13 tokens per second)
llama_perf_context_print:       total time =  118440.29 ms /  1107 tokens
llama_perf_context_print:    graphs reused =         92
 ─ Executing parsed code: ────────────────────────────────────────────────────────────────────────────────────────
  bitcoin_price_usd = 64677.29                                                                                     
  bitcoin_to_egp = currency_converter(amount=bitcoin_price_usd, from_currency="USD", to_currency="EGP")            
  final_answer(bitcoin_to_egp)                                                                                     
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Code execution failed at line 'bitcoin_to_egp = currency_converter(amount=bitcoin_price_usd, from_currency="USD",
to_currency="EGP")' due to: InterpreterError: Forbidden function evaluation: 'currency_converter' is not among the
explicitly allowed tools or defined/imported in the preceding code

[Step 5: Duration 118.48 seconds]

Llama.generate: 5 prefix-match hit, remaining 4019 prompt tokens to eval
llama_perf_context_print:        load time =  135843.17 ms
llama_perf_context_print: prompt eval time =  288225.78 ms /  4019 tokens (   71.72 ms per token,    13.94 tokens per second)
llama_perf_context_print:        eval time =   25016.93 ms /   129 runs   (  193.93 ms per token,     5.16 tokens per second)
llama_perf_context_print:       total time =  313385.50 ms /  4148 tokens
llama_perf_context_print:    graphs reused =        127
Reached max steps.

[Step 6: Duration 313.41 seconds]


==================================================
النتيجة النهائية:
Thought: I need to search for the current Bitcoin price in USD and then convert it to Egyptian Pounds. I will use the 'web_search' tool to find the current Bitcoin price in USD and then use the 'currency_converter' tool to convert it to Egyptian Pounds.
<code>
bitcoin_price_usd = web_search(query="current Bitcoin price in USD")
print("Current Bitcoin price in USD:", bitcoin_price_usd)
bitcoin_price_egp = currency_converter(amount=bitcoin_price_usd, from_currency="USD", to_currency="EGP")
print("Current Bitcoin price in EGP:", bitcoin_price_egp)
</code>
==================================================



تحليل رائع للمخرجات! لقد حققت تقدماً هائلاً: الموديل أصبح الآن قادراً على استخدام الإنترنت، وقراءة النتائج، وكتابة كود بايثون.
لماذا فشل في الخطوة الأخيرة؟ (تحليل اللوج)
هلوسة الأدوات (Tool Hallucination): الموديل ذكي بما يكفي ليعرف أنه يحتاج لتحويل عملة، لكنه "تخيل" وجود أداة جاهزة اسمها currency_converter. وبما أنها غير موجودة في الكود الخاص بك، رفض الوكيل (Agent) تنفيذها.
استخراج الرقم: الموديل حصل على نص طويل من البحث يحتوي على "$64,677.29"، لكنه لم يقم باستخراج الرقم يدوياً ليضربه في 50.
الحل النهائي (التوجيه الصارم + مثال للكود):
سنقوم بتعديل "الطلب" (Query) لنجعل الموديل يفهم أنه يجب أن يستخرج الرقم بنفسه ويقوم بعملية الضرب الحسابية باستخدام بايثون العادي، وليس أداة خارجية.

In [ ]:
# ... (نفس كلاس QwenGGUFModel ونفس الإعدادات السابقة) ...

try:
    local_model = QwenGGUFModel(model_path)

    agent = CodeAgent(
        tools=[DuckDuckGoSearchTool()],
        model=local_model,
        max_steps=5
    )

    # طلب "خوارزمي" دقيق جداً للموديل
    query = """
    Task: Find Bitcoin price and convert to EGP.
    1. Call web_search to find the 'current live price of 1 Bitcoin in USD'.
    2. Look at the search results and find the numeric price (e.g., 64000).
    3. In your Python code, create a variable 'price_usd' with that number.
    4. Calculate 'price_egp = price_usd * 50'.
    5. Use final_answer() to show the result.

    CRITICAL: Do NOT use a 'currency_converter' tool. Just use normal Python math (*).
    """

    print("\n--- بدأ Qwen التنفيذ الذكي ---")
    result = agent.run(query)

    print("\n" + "="*50)
    print(f"النتيجة النهائية:\n{result}")
    print("="*50)

except Exception as e:
    print(f"\nحدث خطأ: {e}")

In [ ]:
from smolagents import CodeAgent, DuckDuckGoSearchTool, Model
from smolagents.models import ChatMessage
from llama_cpp import Llama
from typing import List, Dict, Optional, Any

# بناء الكلاس مع ذاكرة أكبر (8192)
class QwenGGUFModel(Model):
    def __init__(self, model_path: str):
        super().__init__()
        self.model_id = "Qwen2.5-Local"
        print(f"جاري تحميل الموديل بذاكرة مضاعفة...")
        self.llm = Llama(
            model_path=model_path,
            n_ctx=8192,  # زيادة حجم الذاكرة لتجنب خطأ الـ Context
            n_threads=2,
            n_gpu_layers=0
        )

    def generate(self, messages: List[Any], stop_sequences: Optional[List[str]] = None, **kwargs) -> ChatMessage:
        prompt = ""
        for msg in messages:
            role = msg.role
            content_raw = msg.content
            if isinstance(content_raw, list):
                content = "".join([item.get("text", "") for item in content_raw if item.get("type") == "text"])
            else:
                content = content_raw
            prompt += f"<|im_start|>{role}\n{content}<|im_end|>\n"

        prompt += "<|im_start|>assistant\n"

        stops = ["<|im_end|>", "<|endoftext|>"]
        if stop_sequences:
            stops.extend(stop_sequences)

        output = self.llm(
            prompt,
            max_tokens=1024,
            stop=stops,
            echo=False,
            temperature=0.1
        )
        return ChatMessage(role="assistant", content=output['choices'][0]['text'])

# تشغيل النظام
model_path = "/content/Qwen2.5-1.5B-Instruct-Q4_K_M.gguf"

try:
    local_model = QwenGGUFModel(model_path)

    # إضافة تعليمات للوكيل ليكون أكثر ذكاءً في استخدام الأدوات
    agent = CodeAgent(
        tools=[DuckDuckGoSearchTool()],
        model=local_model,
        max_steps=5,
        # توجيه الوكيل للقيام بالحسابات يدوياً
        additional_authorized_imports=["math"]
    )

    # طلب محدد جداً لتجنب التشتت
    query = """
    Step 1: Use web_search to find the current price of 1 Bitcoin in USD.
    Step 2: Take that number and multiply it by 50 to get the price in EGP.
    DO NOT look for a currency converter tool, just do the math in Python.
    """

    print("\n--- بدأ Qwen التنفيذ بذاكرة أكبر ---")
    result = agent.run(query)

    print("\n" + "="*50)
    print(f"النتيجة النهائية:\n{result}")
    print("="*50)

except Exception as e:
    print(f"\nحدث خطأ: {e}")

https://medium.com/@abonia/building-practical-local-ai-agents-with-smolagents-ollama-f92900c51897#id_token=eyJhbGciOiJSUzI1NiIsImtpZCI6ImYxMGY4NzQwNWE5NzljMWRmMzZkZjI2NjA2NzM0ZjMzY2Q4NWMyNzEiLCJ0eXAiOiJKV1QifQ.eyJpc3MiOiJodHRwczovL2FjY291bnRzLmdvb2dsZS5jb20iLCJhenAiOiIyMTYyOTYwMzU4MzQtazFrNnFlMDYwczJ0cDJhMmphbTRsamRjbXMwMHN0dGcuYXBwcy5nb29nbGV1c2VyY29udGVudC5jb20iLCJhdWQiOiIyMTYyOTYwMzU4MzQtazFrNnFlMDYwczJ0cDJhMmphbTRsamRjbXMwMHN0dGcuYXBwcy5nb29nbGV1c2VyY29udGVudC5jb20iLCJzdWIiOiIxMTc4MjQzNDk1MzY2NTY4MTkxOTEiLCJlbWFpbCI6ImxvcHA1MjMzM0BnbWFpbC5jb20iLCJlbWFpbF92ZXJpZmllZCI6dHJ1ZSwibmJmIjoxNzg2MDYxMTIxLCJuYW1lIjoicG9sIGxvcCIsInBpY3R1cmUiOiJodHRwczovL2xoMy5nb29nbGV1c2VyY29udGVudC5jb20vYS9BQ2c4b2NJUXhUTlhjSjVKTjNOeHUzOGlyVDFZRUlkQnlzaWxxU1FITnFFa2xDZThBVkpBdXc9czk2LWMiLCJnaXZlbl9uYW1lIjoicG9sIiwiZmFtaWx5X25hbWUiOiJsb3AiLCJpYXQiOjE3ODYwNjE0MjEsImV4cCI6MTc4NjA2NTAyMSwianRpIjoiNDRlZGJhYjlkODg4NTVmMWNiMjViZDRiZTM1OGIyZjMxNzM3ZWFhMCJ9.C1zmlYgtrHNxhwoye3kzJzmfNO2A_9WNY7-l1s37iiOXrkqxfZWlrp8TYc9mP8_ZYF7OoaKYP_o-UrKD1ReX4lJSeYzlhTvQOsU-B01WCin3DjWAM8CiVrVe4fn4RLl4iYI5Vd5BkHMYwxDRA8_ou78Uy2h2BTFgUG1afMkzNeIEafiH7_LURwlCrkMtpVvWIA0YcLWfvfHq4st0vqKbX2p7KEaD2XDwguPJajaC6q2pVLOZ3TfpWGkr_hkT0npc9HyVop7Y2XodScGNAZXI56ZhQUeU5vfvn_1zgeTGgpbPvOv92ORic3-5blbYQkpZjdLzmB-BOTrB165fxy3Kyw

In [ ]:
ollama pull qwen-2.5

In [ ]:
from smolagents import CodeAgent, DuckDuckGoSearchTool, LiteLLMModel
# Local LLM initialization
model = LiteLLMModel(model_id="ollama_chat/qwen-2.5:14b")
# Define tools
tools = [DuckDuckGoSearchTool()]
# Create the agent
agent = CodeAgent(tools=tools, model=model)
# Example query
result = agent.run("Analyze the top 5 solar battery startups and summarize their funding.")
print(result)

In [ ]:
from smolagents import VectorStoreTool
vector_store = VectorStoreTool(path="local_embeddings/market_data")
agent.tools.append(vector_store)
result = agent.run("Which solar startup has the largest patent portfolio?")

In [ ]:
from smolagents import Memory
agent.memory = Memory(persist=True, path="agent_memory.db")
result = agent.run("Draft a product brief for a solar battery startup using prior research")

In [ ]:
def calculate_roi(cost, revenue):
    return (revenue - cost) / cost
agent.tools.append(calculate_roi)
result = agent.run("Calculate ROI for a startup with $10M cost and $25M revenue")

In [ ]:
Sample Output:

Top 5 Solar Battery Startups:
1. SunVolt - Funding: $50M, Patents: 12, ROI: 1.5
2. EcoCharge - Funding: $35M, Patents: 8, ROI: 1.3
...
Insight: SunVolt leads in patents and ROI; recommend monitoring for investment opportunities.

https://dataisadope.com/blog/build-your-first-local-ai-agent/

Sidebar menu
Search

Get app
Write
Notifications2

Lopp
Home
Library
Profile
Stories
Stats
Following
Medium Staff
Medium Staff
Martin Thissen
Martin Thissen
Find writers and publications to follow.

See suggestions
Welcome Offer
Access to everything. Now 30% off.
Upgrade now

Abonia Sojasingarayar
Abonia Sojasingarayar
Principal Research Scientist | Machine Learning & Ops Engineer | Data Scientist | NLP Engineer | Computer Vision Engineer | AI Analyst

Follow writer

Building Practical Local AI Agents with Smolagents + Ollama
Build Efficient Local Autonomous AI Agents
Abonia Sojasingarayar
Abonia Sojasingarayar

Follow
3 min read
·
Dec 30, 2025
30


1

1





Large language models (LLMs) have transformed how AI systems interact with users. Beyond API calls and prompt engineering, the real frontier is autonomous agents — systems that can reason, plan, execute tasks, and integrate with external tools — all locally, efficiently, and reproducibly.

Two frameworks make this possible: Smolagents, a minimalist agent framework, and Ollama, a local LLM deployment platform. Combined, they allow AI engineers to create robust, fully autonomous agents with fine-grained control over reasoning, memory, and tool integration.

Press enter or click to view image in full size

Image by Author — Agentic Workflow
System Architecture
The typical architecture of a Smolagent-based agent is:

User Input → Agent Planner → Tool Execution → Local LLM → Agent Output
Components:

Planner: Generates stepwise plans based on user instructions.
Executor / Tools: Handles code execution, searches, calculators, APIs, or database queries.
Memory / RAG: Maintains context and integrates private knowledge for multi-step reasoning.
Local LLM (Ollama): Provides inference locally, minimizing latency and preserving privacy.
The design emphasizes modularity, transparency, and extensibility, allowing engineers to swap LLMs, tools, or reasoning pipelines without rewriting the core agent logic.

Practical Setup
1. Install Dependencies
pip install smolagents
Install Ollama and pull a compatible LLM:

brew install ollama
ollama pull qwen-2.5
2. Initialize Agent
from smolagents import CodeAgent, DuckDuckGoSearchTool, LiteLLMModel
# Local LLM initialization
model = LiteLLMModel(model_id="ollama_chat/qwen-2.5:14b")
# Define tools
tools = [DuckDuckGoSearchTool()]
# Create the agent
agent = CodeAgent(tools=tools, model=model)
# Example query
result = agent.run("Analyze the top 5 solar battery startups and summarize their funding.")
print(result)
This setup produces an autonomous agent capable of planning, searching, and reasoning, fully on a local machine.

Advanced Agent Patterns
1. Retrieval-Augmented Generation (RAG)
Integrating a local knowledge base improves accuracy:

from smolagents import VectorStoreTool
vector_store = VectorStoreTool(path="local_embeddings/market_data")
agent.tools.append(vector_store)
result = agent.run("Which solar startup has the largest patent portfolio?")
Enables agents to answer domain-specific questions accurately
Keeps all data local
Works seamlessly with multi-step reasoning
2. Memory & Multi-Step Reasoning
Agents maintain context over multiple interactions:

from smolagents import Memory
agent.memory = Memory(persist=True, path="agent_memory.db")
result = agent.run("Draft a product brief for a solar battery startup using prior research")
Maintains state across sessions
Supports chaining of reasoning steps
Ensures consistency in complex workflows
3. Custom Tool Integration
Agents can execute arbitrary Python functions:

def calculate_roi(cost, revenue):
    return (revenue - cost) / cost
agent.tools.append(calculate_roi)
result = agent.run("Calculate ROI for a startup with $10M cost and $25M revenue")
Enables integration of domain-specific tools, APIs, and computation
Fully sandboxed for safety
Allows automation of complex workflows
Example Use Case: Market Research Agent
Objective: Build an autonomous agent that performs market research on renewable energy startups.

Subscribe to the Medium newsletter
Workflow:

Receive user query: “Analyze top solar battery startups.”
Conduct web search via DuckDuckGoSearchTool.
Query local knowledge base for historical funding, patents, or technical data.
Summarize results and compute financial metrics using custom tools.
Store insights in memory for follow-up queries.
Sample Output:

Top 5 Solar Battery Startups:
1. SunVolt - Funding: $50M, Patents: 12, ROI: 1.5
2. EcoCharge - Funding: $35M, Patents: 8, ROI: 1.3
...
Insight: SunVolt leads in patents and ROI; recommend monitoring for investment opportunities.
This demonstrates a fully autonomous, local agent capable of reasoning, tool usage, and multi-step analysis.

Deployment Considerations
Local Execution: Eliminates dependency on cloud APIs, ensuring privacy and cost-efficiency.
Sandboxing: Any code-executing tools must be sandboxed for security.
Logging & Observability: Capture agent plans, tool usage, and outputs for transparency.
Resource Management: Ollama optimizes memory and inference for large LLMs locally.
Best Practices
Modularize models, tools, and memory for maintainability.
Combine RAG with memory for context-aware, multi-step reasoning.
Test deterministic behavior with fixed seeds for reproducibility.
Always monitor agent outputs in critical workflows.
Conclusion
Smolagents + Ollama provides a practical, local, and extensible platform for building autonomous AI agents. By combining planning, reasoning, tool integration, and memory management, engineers can create robust agents that perform complex tasks locally, without relying on expensive APIs or cloud infrastructure.

Thanks for reading!

Connect with me on Linkedin

Website

Find me on Github

Find me on Youtube

30


1

1




Abonia Sojasingarayar
Written by Abonia Sojasingarayar
1K followers
·
368 following
Principal Research Scientist | Machine Learning & Ops Engineer | Data Scientist | NLP Engineer | Computer Vision Engineer | AI Analyst


Follow
Responses (1)
Lopp
Lopp
What are your thoughts?

Cancel
Respond
John Kintree
John Kintree

Jun 25


Thanks. Local agentic frameworks are critical for building a decentralized platform for local to global governance where people claim issues, submit evidence, and propose and deliberate on solutions.
1


1 reply

Reply

More from Abonia Sojasingarayar
How I Passed the Microsoft Azure AI-900 Exam (2025 Update)
Abonia Sojasingarayar
Abonia Sojasingarayar

·

Sep 8, 2025

How I Passed the Microsoft Azure AI-900 Exam (2025 Update)
Pass the AI-900: Microsoft Azure AI Fundamentals Certification Exam — Prepare, Practice, Take and Pass the Exam
25
3


vLLM Optimization for scalable Scheduling, Batching & Concurrent Inference
Abonia Sojasingarayar
Abonia Sojasingarayar

·

Jun 11

vLLM Optimization for scalable Scheduling, Batching & Concurrent Inference
vLLM Production-grade Optimization — Explained Simply
6
1


BERTScore Explained in 5 minutes
Abonia Sojasingarayar
Abonia Sojasingarayar

·

Jan 15, 2024

BERTScore Explained in 5 minutes
Evaluating Text Generation with BERT: An Overview of BERTScore
147
1


Running Ollama in Google Colab (Free Tier)
Abonia Sojasingarayar
Abonia Sojasingarayar

·

Sep 16, 2024

Running Ollama in Google Colab (Free Tier)
A Step-by-Step Tutorial
260
8


See all from Abonia Sojasingarayar
Recommended from Medium
Stop Wasting LLM Tokens: Building a Self-Updating Codebase Knowledge Graph with OKF
Data Science Collective
In

Data Science Collective

by

Udaykiran Estari

·

Jul 3

Stop Wasting LLM Tokens: Building a Self-Updating Codebase Knowledge Graph with OKF
Google's new Open Knowledge Format (OKF) standardizes context. Here's how to build a pipeline to keep your codebase memory self-updating.

422
10
8


Building a RAG Pipeline for 10M+ Documents With Near-Zero Hallucination
Level Up Coding
In

Level Up Coding

by

Fareed Khan

·

Jun 14

Building a RAG Pipeline for 10M+ Documents With Near-Zero Hallucination
Retrieve, constrain, verify, abstain

1.6K
15
39


How to Use Graphify: Turn Any Folder Into a Knowledge Graph
Agentic Builders
In

Agentic Builders

by

Ana Bildea, PhD

·

Apr 11

How to Use Graphify: Turn Any Folder Into a Knowledge Graph
A step-by-step guide to using Graphify, the open-source tool that builds a queryable knowledge graph

3.4K
25
12


30 Core Agentic Engineering Concepts Every Developer Should Know
Let’s Code Future
In

Let’s Code Future

by

Deep concept

·

Jun 20

30 Core Agentic Engineering Concepts Every Developer Should Know
A simple guide to AI agents, tools, memory, multi-agent systems, and how to build them safely

2.7K
45
55


MCP is Dead
UX Planet
In

UX Planet

by

Nick Babich

·

Apr 6

MCP is Dead
Why you should avoid using MCP in Claude Code and what to use instead

5.6K
296
183


What Is AirLLM and Why It Matters for Running LLMs on Limited Hardware
CodeToDeploy
In

CodeToDeploy

by

Sai Bhargav Rallapalli

·

Feb 10

What Is AirLLM and Why It Matters for Running LLMs on Limited Hardware
Running large language models locally sounds great in theory. In practice, memory becomes the bottleneck long before compute does.

161
2


See more recommendations
Help

Status

About

Careers

Press

Blog

Store

Privacy

Rules

Terms

Text to speech